# US Flight Delays and Cancellations 2015 — analyse complète

**Cours** : restitution de projet data · **Dataset** : vols domestiques américains 2015 (Kaggle / US DOT).

Ce notebook part **uniquement des fichiers bruts** `data/raw/flights.csv`, `data/raw/airlines.csv` et
`data/raw/airports.csv`. Il exécute la chaîne complète : chargement, audit qualité, préparation,
test des trois hypothèses, export des tableaux et génération des données du dashboard HTML.

## Les trois hypothèses testées

| # | Hypothèse | Ce qu'on mesure |
|---|---|---|
| **H1** | Il y a plus de vols en retard pendant les vacances que le reste de l'année. | Taux de vols en retard à l'arrivée, selon la période de l'année. |
| **H2** | Les longs courriers composent la plus grande majorité des retards. | Part des vols >= 1500 miles dans les vols en retard et les minutes de retard, puis contre-analyses distance et compagnies. |
| **H3** | Une petite minorité de liaisons concentre la majorité des annulations. | Concentration des annulations par route, comparée à la concentration du trafic. |

## Mode d'emploi

1. Placer les trois CSV bruts dans `data/raw/`.
2. `Cell > Run All`. Durée indicative : 2 à 4 minutes (le CSV brut fait ~592 Mo).
3. Tous les livrables sont écrits dans `v2/outputs/` et `v2/dashboard/dashboard_data.js`.

> **Règle suivie dans tout le notebook** : aucun chiffre n'est écrit en dur dans le texte.
> Les commentaires et la synthèse finale sont générés à partir des variables calculées,
> pour qu'une réexécution ne puisse jamais contredire le texte.

---
## Sommaire

1. [Environnement et chemins](#1.-Environnement-et-chemins)
2. Chargement des données brutes
3. Audit qualité
4. Préparation des données
5. Panorama général
6. **H1** — « Il y a plus de vols en retard pendant les vacances que le reste de l'année »
7. **H2** — « Les longs courriers composent la plus grande majorité des retards »
8. **H3** — « Une petite minorité de liaisons concentre la majorité des annulations »
9. Figures
10. Exports
11. Synthèse
12. Fichiers produits

---
## 1. Environnement et chemins

On enregistre les versions des librairies pour que le résultat soit traçable, et on localise la racine
du projet en remontant l'arborescence jusqu'à trouver `data/raw`. Le notebook fonctionne donc quel que
soit le dossier depuis lequel Jupyter a été lancé.

In [1]:
import json
import math
import sys
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
T0 = time.time()

# --- racine du projet : on remonte jusqu'àu dossier qui contient data/raw ---
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw" / "flights.csv").exists():
            return p
    raise FileNotFoundError(
        "Impossible de trouver 'data/raw/flights.csv' en remontant depuis " + str(start)
        + ". Placez le notebook dans le projet ou corrigez PROJECT_ROOT manuellement."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW = PROJECT_ROOT / "data" / "raw"
V2 = PROJECT_ROOT / "v2"
OUT = V2 / "outputs"
FIG = OUT / "figures"
DASH = V2 / "dashboard"
ASSETS = V2 / "assets"
for d in (OUT, FIG, DASH, ASSETS):
    d.mkdir(parents=True, exist_ok=True)

def fmt_fr(n: float, dec: int = 0) -> str:
    """Formate un nombre à la francaise : 4 693 et non 4,693."""
    return f"{n:,.{dec}f}".replace(",", " ")

print("Python     ", sys.version.split()[0])
print("pandas     ", pd.__version__)
print("numpy      ", np.__version__)
print("scipy      ", scipy.__version__)
print("matplotlib ", matplotlib.__version__)
print()
print("Racine projet :", PROJECT_ROOT)
print("Données brutes:", RAW)
print("Sorties       :", OUT)

Python      3.12.3
pandas      3.0.5
numpy       2.5.1
scipy       1.18.0
matplotlib  3.11.1

Racine projet : /home/dan/projects/ESGI/4A/T3/Reporting et Restitution/restitution
Données brutes: /home/dan/projects/ESGI/4A/T3/Reporting et Restitution/restitution/data/raw
Sorties       : /home/dan/projects/ESGI/4A/T3/Reporting et Restitution/restitution/v2/outputs


---
## 2. Chargement des données brutes

`flights.csv` fait environ 592 Mo. On ne charge que les colonnes utiles et on impose des types compacts
(`int8`, `int16`, `float32`, `category`) : sans cela, pandas alloue des `int64`/`float64` partout et
l'empreinte mémoire dépasse 2 Go.

Les trois fichiers :

- **`flights.csv`** — une ligne par vol domestique programmé en 2015.
- **`airlines.csv`** — correspondance code IATA compagnie -> nom commercial (14 lignes).
- **`airports.csv`** — 322 aéroports avec ville, état et coordonnées GPS (utilisées pour la carte).

In [2]:
COLUMNS = {
    # identification du vol
    "YEAR": "int16", "MONTH": "int8", "DAY": "int8", "DAY_OF_WEEK": "int8",
    "AIRLINE": "category", "ORIGIN_AIRPORT": "category", "DESTINATION_AIRPORT": "category",
    # horaires et distance
    "SCHEDULED_DEPARTURE": "int16", "DISTANCE": "int16",
    # résultats du vol
    "DEPARTURE_DELAY": "float32", "ARRIVAL_DELAY": "float32",
    "DIVERTED": "int8", "CANCELLED": "int8", "CANCELLATION_REASON": "category",
    # décomposition des minutes de retard
    "AIR_SYSTEM_DELAY": "float32", "SECURITY_DELAY": "float32", "AIRLINE_DELAY": "float32",
    "LATE_AIRCRAFT_DELAY": "float32", "WEATHER_DELAY": "float32",
}

t = time.time()
flights = pd.read_csv(RAW / "flights.csv", usecols=list(COLUMNS), dtype=COLUMNS)
airlines_ref = pd.read_csv(RAW / "airlines.csv")
airports_ref = pd.read_csv(RAW / "airports.csv")

print(f"flights.csv  : {flights.shape[0]:>9,} lignes x {flights.shape[1]} colonnes  "
      f"({flights.memory_usage(deep=True).sum() / 1e6:.0f} Mo, {time.time() - t:.0f} s)")
print(f"airlines.csv : {airlines_ref.shape[0]:>9,} lignes")
print(f"airports.csv : {airports_ref.shape[0]:>9,} lignes")

# Contrôles d'intégrité : si l'un échoue, le fichier source n'est pas celui attendu.
assert len(flights) == 5_819_079, f"Nombre de vols inattendu : {len(flights):,}"
assert set(flights.YEAR.unique()) == {2015}, "Le fichier doit contenir uniquement l'année 2015"
assert len(airlines_ref) == 14, "14 compagnies attendues"
print("\nContrôles d'intégrité : OK")

flights.csv  : 5,819,079 lignes x 19 colonnes  (262 Mo, 11 s)
airlines.csv :        14 lignes
airports.csv :       322 lignes

Contrôles d'intégrité : OK


---
## 3. Audit qualité

Avant toute analyse, on regarde ce qui manque et ce qui est incohérent. Deux points sortent de cet audit
et conditionnent la suite : le traitement des valeurs manquantes de `ARRIVAL_DELAY`, et un problème de
codage des aéroports qui touche un mois entier.

In [3]:
audit = pd.DataFrame({
    "type": flights.dtypes.astype(str),
    "manquants": flights.isna().sum(),
})
audit["manquants_pct"] = (audit.manquants / len(flights) * 100).round(2)
audit.sort_values("manquants", ascending=False)

,type,manquants,manquants_pct
CANCELLATION_REASON,category,5729195,98.46
SECURITY_DELAY,float32,4755640,81.72
WEATHER_DELAY,float32,4755640,81.72
AIR_SYSTEM_DELAY,float32,4755640,81.72
LATE_AIRCRAFT_DELAY,float32,4755640,81.72
AIRLINE_DELAY,float32,4755640,81.72
ARRIVAL_DELAY,float32,105071,1.81
DEPARTURE_DELAY,float32,86153,1.48
DAY,int8,0,0.00
YEAR,int16,0,0.00


### 3.1 Pourquoi `ARRIVAL_DELAY` est manquant

Les 105 071 valeurs manquantes de `ARRIVAL_DELAY` ne sont pas du bruit : ce sont exactement les vols qui
**n'ont jamais atterri comme prévu**, c'est-à-dire les vols annulés et les vols déroutés. C'est structurel,
pas accidentel.

Conséquence directe : écrire `ARRIVAL_DELAY >= 15` sur le dataset complet renvoie `False` pour ces vols,
donc **les compte silencieusement comme des vols à l'heure**. On les exclut explicitement du périmètre des
retards (variable `EST_EXPLOITABLE` construite en section 4).

In [4]:
missing_arrival = flights.ARRIVAL_DELAY.isna()
crosscheck = pd.crosstab(
    np.select([flights.CANCELLED.eq(1), flights.DIVERTED.eq(1)], ["Annulé", "Dérouté"], default="Vol effectué"),
    np.where(missing_arrival, "ARRIVAL_DELAY manquant", "ARRIVAL_DELAY renseigné"),
)
print(crosscheck.to_string())

remaining = int(missing_arrival.sum() - flights.CANCELLED.sum() - flights.DIVERTED.sum())
print(f"\nAnnulés  : {int(flights.CANCELLED.sum()):>7,}")
print(f"Déroutés : {int(flights.DIVERTED.sum()):>7,}")
print(f"Manquants non expliqués par ces deux cas : {remaining:,}")

col_0         ARRIVAL_DELAY manquant  ARRIVAL_DELAY renseigné
row_0                                                        
Annulé                         89884                        0
Dérouté                        15187                        0
Vol effectué                       0                  5714008

Annulés  :  89,884
Déroutés :  15,187
Manquants non expliqués par ces deux cas : 0


### 3.2 Le problème des codes aéroport numériques

`ORIGIN_AIRPORT` et `DESTINATION_AIRPORT` contiennent normalement des codes IATA à 3 lettres (`LAX`, `ORD`...).
Mais une partie des lignes utilise des **identifiants numériques DOT** (`10397`, `13930`...), qui ne sont pas
joignables à `airports.csv`.

La cellule suivante montre que ces lignes ne sont pas réparties au hasard.

In [5]:
orig = flights.ORIGIN_AIRPORT.astype("string")
dest = flights.DESTINATION_AIRPORT.astype("string")
numeric_codes = orig.str.fullmatch(r"\d+").fillna(False) | dest.str.fullmatch(r"\d+").fillna(False)

monthly_breakdown = (
    flights.loc[numeric_codes].groupby("MONTH", observed=True).size()
    .rename("lignes en code numérique").to_frame()
)
monthly_breakdown["part du mois (%)"] = (
    monthly_breakdown["lignes en code numérique"] / flights.groupby("MONTH", observed=True).size() * 100
).round(1)

print(f"Lignes concernées : {int(numeric_codes.sum()):,} soit {numeric_codes.mean() * 100:.2f} % du dataset")
print(f"Mois concernés    : {sorted(flights.loc[numeric_codes, 'MONTH'].unique().tolist())}")
print()
print(monthly_breakdown.to_string())

Lignes concernées : 486,165 soit 8.35 % du dataset
Mois concernés    : [10]

       lignes en code numérique  part du mois (%)
MONTH                                            
10                       486165             100.0


In [6]:
# Que représente octobre dans le phénomène étudié par H3 ?
oct_cancellations = int(flights.loc[flights.MONTH.eq(10), "CANCELLED"].sum())
total_cancellations = int(flights.CANCELLED.sum())
oct_rate = flights.loc[flights.MONTH.eq(10), "CANCELLED"].mean() * 100
annual_rate = flights.CANCELLED.mean() * 100

print(f"Octobre = {numeric_codes.mean() * 100:.2f} % des vols de l'année")
print(f"Octobre = {oct_cancellations:,} annulations sur {total_cancellations:,}, soit {oct_cancellations / total_cancellations * 100:.2f} % du total")
print(f"Taux d'annulation en octobre : {oct_rate:.2f} %  (moyenne annuelle : {annual_rate:.2f} %)")

Octobre = 8.35 % des vols de l'année
Octobre = 2,454 annulations sur 89,884, soit 2.73 % du total
Taux d'annulation en octobre : 0.50 %  (moyenne annuelle : 1.54 %)


**Décision et conséquence assumée.** La totalité des codes numériques est concentrée sur **octobre 2015** :
ce mois utilise un autre référentiel d'aéroports que les onze autres. Comme H3 raisonne route par route,
on ne peut pas mélanger deux référentiels — une même liaison apparaîtrait deux fois sous deux noms différents.

On restreint donc **H3 aux onze mois en codes IATA**. Filtrer sur `code IATA lisible` revient donc
exactement à **exclure octobre**, ce qui doit être dit explicitement plutôt que subi.

Ce que ca coûte : octobre pèse 8,35 % des vols mais seulement 2,73 % des annulations, avec un taux
d'annulation de 0,50 % contre 1,54 % sur l'année. C'est **le mois le plus calme de 2015**. Son exclusion
retiré donc peu du phénomène étudié, et va plutôt dans le sens d'une sous-estimation de la concentration.

**H1 et H2 conservent les douze mois** : elles raisonnent sur des dates et des compagnies, pas sur des routes,
et ne sont donc pas affectées par ce problème de codage.

---
## 4. Préparation des données

On construit les variables d'analyse. Chaque variable dérivée répond à un besoin précis des hypothèses ;
elles sont récapitulées dans le dictionnaire en fin de section.

In [7]:
flights["DATE"] = pd.to_datetime(dict(year=flights.YEAR, month=flights.MONTH, day=flights.DAY), errors="coerce")
flights["JOUR_ANNEE"] = flights.DATE.dt.dayofyear.astype("int16")
flights["HEURE_DEP"] = ((flights.SCHEDULED_DEPARTURE // 100) % 24).astype("int8")

# Périmètre des retards : un vol annulé ou dérouté n'a pas de retard d'arrivée mesurable.
flights["EST_EXPLOITABLE"] = flights.CANCELLED.eq(0) & flights.DIVERTED.eq(0)

# Définition du retard, alignée sur le standard US DOT : 15 minutes ou plus à l'arrivée.
DELAY_THRESHOLD = 15
flights["EST_EN_RETARD"] = flights.ARRIVAL_DELAY.ge(DELAY_THRESHOLD) & flights.EST_EXPLOITABLE

# Vol long-courrier domestique (H2) : il n'y a pas de long-courrier au sens international dans ce
# dataset 100 % domestique. On prend 1500 miles, seuil au-delà duquel un vol traverse le pays.
LONG_HAUL_THRESHOLD = 1500
flights["EST_VOL_LONG"] = flights.DISTANCE.ge(LONG_HAUL_THRESHOLD)

# Route et périmètre exploitable pour H3 (cf. section 3.2).
flights["ROUTE_LISIBLE"] = ~numeric_codes
flights["ROUTE"] = orig.where(flights.ROUTE_LISIBLE) + " -> " + dest.where(flights.ROUTE_LISIBLE)

assert flights.DATE.isna().sum() == 0, "Des dates n'ont pas pu être construites"
assert flights.loc[flights.EST_EXPLOITABLE, "ARRIVAL_DELAY"].isna().sum() == 0, \
    "Il reste des retards manquants dans le périmètre exploitable"
print("Variables construites, contrôles OK.")

Variables construites, contrôles OK.


### 4.1 Fenêtres de forte mobilité (variable centrale de H1)

Le dataset ne contient **aucune colonne vacances** : il faut la fabriquer. On définit quatre fenêtres de
forte mobilité du calendrier américain, chacune bornée explicitement pour être reproductible et discutable :

| Fenêtre | Dates 2015 | Nature |
|---|---|---|
| Vacances d'été | 1er juin -> 31 août | saison longue, hausse durable du trafic loisir |
| Nouvel an | 1er -> 5 janvier | pic court, retours de fêtes |
| Thanksgiving | 20 -> 30 novembre | pic court, plus gros week-end de déplacement US |
| Noël / fin d'année | 18 -> 31 décembre | pic court, départs de fêtes |

C'est un **proxy**, pas une vérité : il sera confronte à la réalité des données en section 6.

In [8]:
d = flights.DATE
HOLIDAY_WINDOWS = {
    "Vacances d'été":     (flights.MONTH.isin([6, 7, 8]),                    "juin -> août"),
    "Nouvel an":          (flights.MONTH.eq(1) & flights.DAY.le(5),             "1 -> 5 janvier"),
    "Thanksgiving":       (d.between("2015-11-20", "2015-11-30"),         "20 -> 30 novembre"),
    "Noël / fin d'année": (d.between("2015-12-18", "2015-12-31"),         "18 -> 31 décembre"),
}

flights["EST_VACANCES"] = np.logical_or.reduce([m for m, _ in HOLIDAY_WINDOWS.values()])
flights["FENETRE"] = np.select(
    [m for m, _ in HOLIDAY_WINDOWS.values()], list(HOLIDAY_WINDOWS), default="Hors vacances",
)
# Pics de fin d'année seuls : utile pour isoler l'effet saison de l'effet fête.
flights["EST_PIC_FETE"] = np.logical_or.reduce([HOLIDAY_WINDOWS[k][0] for k in
                                             ("Nouvel an", "Thanksgiving", "Noël / fin d'année")])

print(flights.FENETRE.value_counts().to_string())

FENETRE
Hors vacances         3823210
Vacances d'été        1535151
Noël / fin d'année     216004
Thanksgiving           165689
Nouvel an               79025


In [9]:
DATA_DICTIONARY = pd.DataFrame([
    ("DATE",            "date",    "Date du vol, reconstruite depuis YEAR/MONTH/DAY"),
    ("EST_EXPLOITABLE", "bool",    "Vol ni annulé ni dérouté : seul périmètre où le retard existe"),
    ("EST_EN_RETARD",   "bool",    f"Arrivée avec >= {DELAY_THRESHOLD} min de retard (standard US DOT), sur vols exploitables"),
    ("EST_VOL_LONG",    "bool",    f"Distance >= {LONG_HAUL_THRESHOLD} miles : long-courrier domestique (H2)"),
    ("EST_VACANCES",    "bool",    "Vol dans l'une des 4 fenêtres de forte mobilité (H1)"),
    ("FENETRE",         "texte",   "Nom de la fenêtre de mobilité, ou 'Hors vacances'"),
    ("EST_PIC_FETE",    "bool",    "Fenêtre courte de fête uniquement (hors été), pour isoler l'effet fête"),
    ("ROUTE",           "texte",   "ORIGINE -> DESTINATION, uniquement si les deux codes sont IATA (H3)"),
    ("ROUTE_LISIBLE",   "bool",    "Les deux aéroports ont un code IATA exploitable (exclut octobre)"),
    ("HEURE_DEP",       "int",     "Heure de départ programmée, 0 à 23"),
], columns=["variable", "type", "definition"])
DATA_DICTIONARY

,variable,type,definition
0,DATE,date,"Date du vol, reconstruite depuis YEAR/MONTH/DAY"
1,EST_EXPLOITABLE,bool,Vol ni annulé ni dérouté : seul périmètre où l...
2,EST_EN_RETARD,bool,Arrivée avec >= 15 min de retard (standard US ...
3,EST_VOL_LONG,bool,Distance >= 1500 miles : long-courrier domesti...
4,EST_VACANCES,bool,Vol dans l'une des 4 fenêtres de forte mobilit...
5,FENETRE,texte,"Nom de la fenêtre de mobilité, ou 'Hors vacances'"
6,EST_PIC_FETE,bool,"Fenêtre courte de fête uniquement (hors été), ..."
7,ROUTE,texte,"ORIGINE -> DESTINATION, uniquement si les deux..."
8,ROUTE_LISIBLE,bool,Les deux aéroports ont un code IATA exploitabl...
9,HEURE_DEP,int,"Heure de départ programmée, 0 à 23"


---
## 5. Panorama général

Avant de tester les hypothèses, on pose les ordres de grandeur : ce sont les repères auxquels tous les
résultats suivants seront comparés.

In [10]:
completed_flights = flights[flights.EST_EXPLOITABLE]

GLOBAL_DELAY_RATE = completed_flights.EST_EN_RETARD.mean() * 100
GLOBAL_CANCEL_RATE = flights.CANCELLED.mean() * 100
GLOBAL_MEAN_DELAY = completed_flights.ARRIVAL_DELAY.mean()

kpi = {
    "vols_programmes": len(flights),
    "vols_exploitables": int(flights.EST_EXPLOITABLE.sum()),
    "vols_annules": int(flights.CANCELLED.sum()),
    "vols_deroutes": int(flights.DIVERTED.sum()),
    "compagnies": int(flights.AIRLINE.nunique()),
    # Uniquement les codes IATA : les identifiants numériques d'octobre gonfleraient artificiellement
    # ce compte en faisant apparaître chaque aéroport une seconde fois sous un autre code.
    "aeroports": int(pd.concat([orig[flights.ROUTE_LISIBLE], dest[flights.ROUTE_LISIBLE]]).nunique()),
    "routes_lisibles": int(flights.loc[flights.ROUTE_LISIBLE, "ROUTE"].nunique()),
    "taux_retard_pct": round(GLOBAL_DELAY_RATE, 2),
    "taux_annulation_pct": round(GLOBAL_CANCEL_RATE, 2),
    "retard_moyen_min": round(float(GLOBAL_MEAN_DELAY), 2),
    "retard_median_min": round(float(completed_flights.ARRIVAL_DELAY.median()), 2),
}
for k, v in kpi.items():
    print(f"{k:22s} {v:>12,}" if isinstance(v, int) else f"{k:22s} {v:>12}")

vols_programmes           5,819,079
vols_exploitables         5,714,008
vols_annules                 89,884
vols_deroutes                15,187
compagnies                       14
aeroports                       322
routes_lisibles               4,693
taux_retard_pct               18.61
taux_annulation_pct            1.54
retard_moyen_min               4.41
retard_median_min              -5.0


Premier constat à garder en tête : le retard **médian** est négatif alors que le retard **moyen** est positif.
La majorité des vols arrivent en avance, et la moyenne est tirée vers le haut par une minorité de vols
très en retard. C'est pourquoi tout le notebook raisonne sur un **taux de vols en retard** (part de vols
au-delà de 15 minutes) plutôt que sur une moyenne, qui serait trompeuse.

In [11]:
# Séries temporelles : mensuelle (lecture) et journalière (détection des pics)
monthly = flights.groupby("MONTH", observed=True).agg(
    vols=("CANCELLED", "size"), taux_annulation=("CANCELLED", "mean")).reset_index()
monthly = monthly.merge(
    completed_flights.groupby("MONTH", observed=True).EST_EN_RETARD.mean().rename("taux_retard").reset_index(),
    on="MONTH")
monthly["taux_annulation"] *= 100
monthly["taux_retard"] *= 100

daily = flights.groupby("DATE", observed=True).agg(
    vols=("CANCELLED", "size"), taux_annulation=("CANCELLED", "mean")).reset_index()
daily = daily.merge(
    completed_flights.groupby("DATE", observed=True).EST_EN_RETARD.mean().rename("taux_retard").reset_index(),
    on="DATE")
daily["taux_annulation"] *= 100
daily["taux_retard"] *= 100
daily["jour_annee"] = daily.DATE.dt.dayofyear
daily["fenetre"] = flights.groupby("DATE", observed=True).FENETRE.first().values

print(monthly.round(2).to_string(index=False))
print(f"\nJours couverts : {len(daily)}")

 MONTH   vols  taux_annulation  taux_retard
     1 469968             2.55        21.00
     2 429191             4.78        23.35
     3 504312             2.18        19.40
     4 485151             0.93        17.16
     5 496993             1.15        18.31
     6 503897             1.81        23.48
     7 520718             0.92        20.92
     8 510536             0.99        18.67
     9 464946             0.45        13.00
    10 486165             0.50        12.44
    11 467972             0.98        15.26
    12 479230             1.68        20.60

Jours couverts : 365


In [12]:
WORST_DELAY_DAYS = daily.nlargest(5, "taux_retard")[["DATE", "vols", "taux_retard", "taux_annulation"]]
WORST_CANCEL_DAYS = daily.nlargest(5, "taux_annulation")[["DATE", "vols", "taux_retard", "taux_annulation"]]
print("Les 5 pires journées en retards :")
print(WORST_DELAY_DAYS.round(2).to_string(index=False))
print("\nLes 5 pires journées en annulations :")
print(WORST_CANCEL_DAYS.round(2).to_string(index=False))

Les 5 pires journées en retards :
      DATE  vols  taux_retard  taux_annulation
2015-01-04 16352        49.30             2.65
2015-01-03 15434        45.22             2.14
2015-12-30 16260        43.65             1.73
2015-03-01 15171        41.31             9.98
2015-12-29 16199        39.92             4.23

Les 5 pires journées en annulations :
      DATE  vols  taux_retard  taux_annulation
2015-01-27 15155         7.13            19.03
2015-02-02 15975        28.88            17.52
2015-03-05 16622        31.21            17.19
2015-02-01 13406        21.83            14.76
2015-12-28 16312        36.38            13.35


/tmp/ipykernel_625017/1588871706.py:4: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(WORST_DELAY_DAYS.round(2).to_string(index=False))
/tmp/ipykernel_625017/1588871706.py:6: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(WORST_CANCEL_DAYS.round(2).to_string(index=False))


Ces deux classements ne se recoupent pas, et c'est une information à part entière :

- Les pires journées de **retard** sont des journées de **retour de fêtes** (début janvier, fin décembre).
- Les pires journées d'**annulation** sont des journées de **tempête hivernale** (fin janvier, février, début mars).

Retard et annulation ne sont donc pas le même phénomène, et ne répondent pas aux mêmes causes. C'est la
grille de lecture qui structure H1 (retards, effet calendaire) et H3 (annulations, effet météo/réseau).

In [13]:
# D'où viennent les minutes de retard, et pourquoi annulé-t-on un vol ?
CAUSE_COLS = ["LATE_AIRCRAFT_DELAY", "AIRLINE_DELAY", "AIR_SYSTEM_DELAY", "WEATHER_DELAY", "SECURITY_DELAY"]
CAUSE_LABELS = {
    "LATE_AIRCRAFT_DELAY": "Avion arrivé en retard", "AIRLINE_DELAY": "Compagnie",
    "AIR_SYSTEM_DELAY": "Contrôle aérien / aéroport", "WEATHER_DELAY": "Météo", "SECURITY_DELAY": "Sécurité",
}
cause_minutes = completed_flights.loc[completed_flights.EST_EN_RETARD, CAUSE_COLS].sum().astype("float64")
delay_causes = (cause_minutes / cause_minutes.sum() * 100).round(2).rename("part_minutes_pct").reset_index()
delay_causes["cause"] = delay_causes["index"].map(CAUSE_LABELS)

CANCEL_LABELS = {"A": "Compagnie", "B": "Météo", "C": "Contrôle aérien / aéroport", "D": "Sécurité"}
cancel_causes = (flights.loc[flights.CANCELLED.eq(1), "CANCELLATION_REASON"]
                .value_counts().rename("vols").reset_index())
cancel_causes.columns = ["code", "vols"]
cancel_causes["cause"] = cancel_causes.code.map(CANCEL_LABELS)
cancel_causes["part_pct"] = cancel_causes.vols / cancel_causes.vols.sum() * 100

print("Répartition des minutes de retard :")
print(delay_causes[["cause", "part_minutes_pct"]].round(2).to_string(index=False))
print("\nMotifs d'annulation :")
print(cancel_causes[["cause", "vols", "part_pct"]].round(2).to_string(index=False))

Répartition des minutes de retard :
                     cause  part_minutes_pct
    Avion arrivé en retard             39.84
                 Compagnie             32.20
Contrôle aérien / aéroport             22.88
                     Météo              4.95
                  Sécurité              0.13

Motifs d'annulation :
                     cause  vols  part_pct
                     Météo 48851     54.35
                 Compagnie 25262     28.11
Contrôle aérien / aéroport 15749     17.52
                  Sécurité    22      0.02


---
## 6. H1 — « Il y a plus de vols en retard pendant les vacances que le reste de l'année »

On procède en trois temps volontairement séparés :

1. **Le test naïf** — celui qu'on ferait spontanément : vacances contre hors vacances.
2. **La décomposition** — qui montre que l'agrégat cache des comportements opposés.
3. **Le test contrôle** — qui compare chaque fenêtre à une référence comparable, et donne la vraie réponse.

### 6.1 Test naïf : vacances contre hors vacances

In [14]:
h1_naive = completed_flights.groupby(flights.EST_VACANCES, observed=True).agg(
    vols=("EST_EN_RETARD", "size"),
    taux_retard=("EST_EN_RETARD", "mean"),
    retard_moyen=("ARRIVAL_DELAY", "mean"),
    retard_median=("ARRIVAL_DELAY", "median"),
).reset_index()
h1_naive["periode"] = np.where(h1_naive.EST_VACANCES, "Périodes de vacances", "Hors vacances")
h1_naive["taux_retard"] *= 100

HOLIDAY_RATE = float(h1_naive.loc[h1_naive.EST_VACANCES, "taux_retard"].iloc[0])
BASELINE_RATE = float(h1_naive.loc[~h1_naive.EST_VACANCES, "taux_retard"].iloc[0])
NAIVE_GAP = HOLIDAY_RATE - BASELINE_RATE

h1_naive[["periode", "vols", "taux_retard", "retard_moyen", "retard_median"]].round(2)

,periode,vols,taux_retard,retard_moyen,retard_median
0,Hors vacances,3753092,16.89,2.74,-5.0
1,Périodes de vacances,1960916,21.91,7.60,-4.0


In [15]:
def chi2_test(group_mask, target_mask, base):
    """Chi-2 d'independance + taille d'effet. Renvoie un dict lisible.

    Sur plusieurs millions de lignes, la p-value n'apprend plus rien : elle est toujours
    inférieure à 0,05. On renvoie donc systématiquement le V de Cramer et le risque relatif,
    qui eux mesurent l'intensite de la relation et non seulement son existence.
    """
    table = pd.crosstab(group_mask.loc[base.index], target_mask.loc[base.index])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    n = table.values.sum()
    v = math.sqrt(chi2 / (n * (min(table.shape) - 1)))
    p1 = table.iloc[1, 1] / table.iloc[1].sum()
    p0 = table.iloc[0, 1] / table.iloc[0].sum()
    return {"chi2": round(float(chi2), 1), "p_value": float(p), "dof": int(dof),
            "cramer_v": round(v, 4), "risque_relatif": round(float(p1 / p0), 3), "n": int(n)}

H1_TEST = chi2_test(flights.EST_VACANCES, flights.EST_EN_RETARD, completed_flights)

print(f"Taux en période de vacances : {HOLIDAY_RATE:.2f} %")
print(f"Taux hors vacances          : {BASELINE_RATE:.2f} %")
print(f"Écart                       : {NAIVE_GAP:+.2f} points")
print(f"Risque relatif              : x{H1_TEST['risque_relatif']}")
print()
print(f"chi2 = {H1_TEST['chi2']:,.1f}   p = {H1_TEST['p_value']:.3g}   V de Cramer = {H1_TEST['cramer_v']}")
print(f"Interprétation du V : {'effet faible' if H1_TEST['cramer_v'] < 0.1 else 'effet modéré'} "
      f"(seuils usuels : 0,1 faible / 0,3 modéré / 0,5 fort)")

Taux en période de vacances : 21.91 %
Taux hors vacances          : 16.89 %
Écart                       : +5.03 points
Risque relatif              : x1.298

chi2 = 21,472.5   p = 0   V de Cramer = 0.0613
Interprétation du V : effet faible (seuils usuels : 0,1 faible / 0,3 modéré / 0,5 fort)


**Ce que dit — et ne dit pas — ce premier résultat.** L'écart est réel et va dans le sens de l'hypothèse.
Mais la p-value ne doit convaincre personne ici : avec 5,7 millions d'observations, **n'importe quel écart,
même minuscule, ressort comme significatif**. Le V de Cramer, lui, indique une association faible.

Surtout, l'agrégat « vacances » mélange quatre fenêtres très différentes. On les sépare avant de conclure.

### 6.2 Décomposition : l'agrégat cache des comportements opposés

In [16]:
rows_acc = []
for name, (mask, date_label) in HOLIDAY_WINDOWS.items():
    subset = completed_flights[mask.loc[completed_flights.index]]
    rows_acc.append({
        "fenetre": name, "dates": date_label, "vols": len(subset),
        "taux_retard": subset.EST_EN_RETARD.mean() * 100,
        "retard_moyen": subset.ARRIVAL_DELAY.mean(),
    })
non_holiday = completed_flights[~flights.EST_VACANCES.loc[completed_flights.index]]
rows_acc.append({"fenetre": "Hors vacances", "dates": "reste de l'année", "vols": len(non_holiday),
               "taux_retard": non_holiday.EST_EN_RETARD.mean() * 100, "retard_moyen": non_holiday.ARRIVAL_DELAY.mean()})

h1_detail = pd.DataFrame(rows_acc).sort_values("taux_retard", ascending=False)
h1_detail["ecart_vs_hors_vacances"] = h1_detail.taux_retard - BASELINE_RATE
h1_detail["part_du_volume_vacances_pct"] = np.where(
    h1_detail.fenetre.eq("Hors vacances"), np.nan,
    h1_detail.vols / h1_detail.loc[h1_detail.fenetre.ne("Hors vacances"), "vols"].sum() * 100)
h1_detail.round(2).to_string(index=False)

"           fenetre             dates    vols  taux_retard  retard_moyen  ecart_vs_hors_vacances  part_du_volume_vacances_pct\n         Nouvel an    1 -> 5 janvier   76931        36.17     18.559999                   19.28                         3.92\nNoël / fin d'année 18 -> 31 décembre  209038        28.82     14.320000                   11.93                        10.66\n    Vacances d'été      juin -> août 1511187        21.01      6.860000                    4.12                        77.07\n     Hors vacances  reste de l'année 3753092        16.89      2.740000                    0.00                          NaN\n      Thanksgiving 20 -> 30 novembre  163760        14.73      0.680000                   -2.16                         8.35"

In [17]:
SUMMER_WEIGHT = float(h1_detail.loc[h1_detail.fenetre.eq("Vacances d'été"), "part_du_volume_vacances_pct"].iloc[0])
THANKSGIVING_RATE = float(h1_detail.loc[h1_detail.fenetre.eq("Thanksgiving"), "taux_retard"].iloc[0])

print(f"L'été représente {SUMMER_WEIGHT:.1f} % du volume classé 'vacances'.")
print(f"L'agrégat 'vacances' est donc à {SUMMER_WEIGHT:.0f} % un agrégat 'été'.")
print()
print(f"Thanksgiving : {THANKSGIVING_RATE:.2f} % de retards, soit {THANKSGIVING_RATE - BASELINE_RATE:+.2f} points "
      f"vs hors vacances -> la fenêtre est {'SOUS' if THANKSGIVING_RATE < BASELINE_RATE else 'au-dessus de'} la référence.")

L'été représente 77.1 % du volume classé 'vacances'.
L'agrégat 'vacances' est donc à 77 % un agrégat 'été'.

Thanksgiving : 14.73 % de retards, soit -2.16 points vs hors vacances -> la fenêtre est SOUS la référence.


**Le problème est maintenant visible.** Thanksgiving — le plus gros week-end de déplacement des États-Unis —
affiche **moins** de retards que le reste de l'année. Une fenêtre sur quatre contredit donc frontalement
l'hypothèse, et l'agrégat le masque parce qu'il est dominé aux trois quarts par l'été.

Il y a en plus un biais de comparaison : comparer décembre à la moyenne annuelle, c'est comparer un mois
d'hiver à une référence qui contient le printemps et l'automne. On mesure alors la **saison**, pas les vacances.

### 6.3 Test contrôle : chaque fenêtre contre une référence comparable

Correction du biais : chaque fenêtre courte est comparée **au reste de son propre mois**, ce qui neutralisé
la saison et la météo moyenne du mois. L'été, qui couvre des mois entiers, ne peut pas être traité ainsi :
on le compare au reste de l'année hors fenêtres de fête, ce qui en fait une mesure d'effet saisonnier.

In [18]:
def net_effect(name, mask):
    """Compare une fenêtre à une référence comparable et renvoie l'écart net en points."""
    if name == "Vacances d'été":
        ref_mask = ~flights.MONTH.isin([6, 7, 8]) & ~flights.EST_PIC_FETE
        ref_label = "reste de l'année hors fêtes"
        kind = "effet saisonnier"
    else:
        months = sorted(flights.loc[mask, "MONTH"].unique().tolist())
        ref_mask = flights.MONTH.isin(months) & ~mask
        ref_label = "reste du même mois"
        kind = "effet fenêtre"

    in_window = completed_flights[mask.loc[completed_flights.index]]
    ref = completed_flights[ref_mask.loc[completed_flights.index]]
    base = completed_flights[(mask | ref_mask).loc[completed_flights.index]]
    test = chi2_test(mask, flights.EST_EN_RETARD, base)
    return {
        "fenetre": name, "nature": kind, "reference": ref_label,
        "vols": len(in_window), "taux_fenetre": in_window.EST_EN_RETARD.mean() * 100,
        "taux_reference": ref.EST_EN_RETARD.mean() * 100,
        "effet_net_points": in_window.EST_EN_RETARD.mean() * 100 - ref.EST_EN_RETARD.mean() * 100,
        "risque_relatif": test["risque_relatif"], "cramer_v": test["cramer_v"], "p_value": test["p_value"],
    }

h1_controlled = pd.DataFrame([net_effect(n, m) for n, (m, _) in HOLIDAY_WINDOWS.items()])
h1_controlled = h1_controlled.sort_values("effet_net_points", ascending=False).reset_index(drop=True)
h1_controlled.round(3).to_string(index=False)

"           fenetre           nature                   reference    vols  taux_fenetre  taux_reference  effet_net_points  risque_relatif  cramer_v  p_value\n         Nouvel an    effet fenêtre          reste du même mois   76931        36.171          17.924            18.248           2.018     0.168      0.0\nNoël / fin d'année    effet fenêtre          reste du même mois  209038        28.817          14.015            14.803           2.056     0.182      0.0\n    Vacances d'été effet saisonnier reste de l'année hors fêtes 1511187        21.009          16.887             4.122           1.244     0.048      0.0\n      Thanksgiving    effet fenêtre          reste du même mois  163760        14.731          15.555            -0.823           0.947     0.011      0.0"

In [19]:
for _, r in h1_controlled.iterrows():
    sens = "AUGMENTE" if r.effet_net_points > 0.5 else ("DIMINUE" if r.effet_net_points < -0.5 else "n'a pas d'effet sur")
    print(f"{r.fenetre:20s} {r.taux_fenetre:5.2f} % vs {r.taux_reference:5.2f} % ({r.reference:28s})"
          f" -> {r.effet_net_points:+6.2f} pt  : {sens} les retards")

Nouvel an            36.17 % vs 17.92 % (reste du même mois          ) -> +18.25 pt  : AUGMENTE les retards
Noël / fin d'année   28.82 % vs 14.01 % (reste du même mois          ) -> +14.80 pt  : AUGMENTE les retards
Vacances d'été       21.01 % vs 16.89 % (reste de l'année hors fêtes ) ->  +4.12 pt  : AUGMENTE les retards
Thanksgiving         14.73 % vs 15.55 % (reste du même mois          ) ->  -0.82 pt  : DIMINUE les retards


### 6.4 Conclusion H1

In [20]:
holiday_peaks = h1_controlled[h1_controlled.nature.eq("effet fenêtre")]
positive_peaks = holiday_peaks[holiday_peaks.effet_net_points > 0.5]
negative_peaks = holiday_peaks[holiday_peaks.effet_net_points < -0.5]
summer_effect = float(h1_controlled.loc[h1_controlled.fenetre.eq("Vacances d'été"), "effet_net_points"].iloc[0])
top_effect = h1_controlled.iloc[0]

H1_STATUS = "Confirmée, mais à préciser"
H1_SUMMARY = (
    f"L'écart brut est de {NAIVE_GAP:+.2f} points ({HOLIDAY_RATE:.2f} % contre {BASELINE_RATE:.2f} %), "
    f"mais cet agrégat est dominé à {SUMMER_WEIGHT:.0f} % par l'été et masque des comportements opposés. "
    f"Une fois chaque fenêtre comparée à une référence comparable, l'effet le plus fort est "
    f"{top_effect.fenetre} ({top_effect.effet_net_points:+.2f} points vs {top_effect.reference}), "
    f"l'été ne pèse que {summer_effect:+.2f} points, et Thanksgiving est "
    f"{'negatif' if THANKSGIVING_RATE < BASELINE_RATE else 'positif'} "
    f"({float(holiday_peaks.loc[holiday_peaks.fenetre.eq('Thanksgiving'), 'effet_net_points'].iloc[0]):+.2f} point). "
    f"Ce ne sont donc pas 'les vacances' qui créent du retard, mais les pics courts de fin d'année."
)
print(H1_STATUS.upper()); print(); print(H1_SUMMARY)

CONFIRMÉE, MAIS À PRÉCISER

L'écart brut est de +5.03 points (21.91 % contre 16.89 %), mais cet agrégat est dominé à 77 % par l'été et masque des comportements opposés. Une fois chaque fenêtre comparée à une référence comparable, l'effet le plus fort est Nouvel an (+18.25 points vs reste du même mois), l'été ne pèse que +4.12 points, et Thanksgiving est negatif (-0.82 point). Ce ne sont donc pas 'les vacances' qui créent du retard, mais les pics courts de fin d'année.


**Réponse à l'hypothèse : confirmée, mais pas pour la raison qu'elle suggère.**

L'effet existe et il est important, mais il est porte par les **pics courts de fin d'année** (Nouvel an, Noel),
pas par « les vacances » en général. L'été produit un effet modéré, et Thanksgiving n'en produit aucun.

L'explication tient à la nature des pics : Noel et Nouvel an cumulent un trafic record **et** la météo
hivernale, sur un réseau déjà saturé. Thanksgiving concentre son trafic sur peu de jours mais bénéficie
d'une météo de novembre plus clémente, et les compagnies y programment des marges horaires renforcées.

**Ce que ca implique pour le dashboard** : afficher un simple « vacances vs hors vacances » serait trompeur.
Il faut montrer la courbe journalière, où l'on voit les pics réels, et l'effet net fenêtre par fenêtre.

---
## 7. H2 — « Les longs courriers composent la plus grande majorité des retards »

Le dataset est intégralement domestique : il n'y a pas de long-courrier au sens international. On retient
**1500 miles ou plus**, seuil au-delà duquel un vol traverse une bonne partie du pays (Boston-Denver,
Chicago-Los Angeles). La sensibilité à ce choix est testée en 7.1.

L'hypothèse est une affirmation de **composition** : elle prétend que la majorité des retards est portée
par les vols longs. On procède en trois temps :

1. **Le test direct** — quelle part des vols en retard, et des minutes de retard, vient des vols longs ?
2. **Le mécanisme** — ce que la distance fait réellement au retard d'arrivée.
3. **La contre-analyse par compagnie** — même les compagnies spécialisées en vols longs sont-elles plus en retard ?

### 7.1 Test direct : la part des retards portée par les vols longs

« Composer les retards » peut se mesurer de deux manieres : en **nombre de vols en retard** (chaque vol en
retard compte pour 1) et en **minutes de retard cumulées** (un vol très en retard pèse plus lourd). On
calcule les deux, avec la **part du trafic** comme référence, et on teste deux autres seuils de distance
pour vérifier que le résultat ne dépend pas du choix de 1500 miles.

In [21]:
delayed_minutes = flights.ARRIVAL_DELAY.where(flights.EST_EN_RETARD, 0.0).astype("float64")
TOTAL_MINUTES = float(delayed_minutes.sum())
N_DELAYED = int(flights.EST_EN_RETARD.sum())

compo_rows = []
for threshold in (1000, LONG_HAUL_THRESHOLD, 2000):
    long_mask = flights.DISTANCE.ge(threshold)
    compo_rows.append({
        "seuil_miles": threshold,
        "part_trafic_pct": long_mask[flights.EST_EXPLOITABLE].mean() * 100,
        "part_vols_retard_pct": (long_mask & flights.EST_EN_RETARD).sum() / N_DELAYED * 100,
        "part_minutes_pct": delayed_minutes[long_mask].sum() / TOTAL_MINUTES * 100,
    })
h2_compo = pd.DataFrame(compo_rows)

ref = h2_compo[h2_compo.seuil_miles.eq(LONG_HAUL_THRESHOLD)].iloc[0]
H2_COMPO = {
    "part_trafic_pct": round(float(ref.part_trafic_pct), 2),
    "part_vols_retard_pct": round(float(ref.part_vols_retard_pct), 2),
    "part_minutes_pct": round(float(ref.part_minutes_pct), 2),
    "sensibilite": h2_compo.round(2).to_dict("records"),
}

print(f"Vols en retard : {N_DELAYED:,} | minutes de retard cumulées : {TOTAL_MINUTES / 1e6:.1f} M")
print()
print(h2_compo.round(2).to_string(index=False))
print(f"\nAu seuil de {LONG_HAUL_THRESHOLD} miles : {ref.part_vols_retard_pct:.1f} % des vols en retard et "
      f"{ref.part_minutes_pct:.1f} % des minutes de retard viennent des vols longs, "
      f"pour {ref.part_trafic_pct:.1f} % du trafic.")

Vols en retard : 1,063,439 | minutes de retard cumulées : 62.7 M

 seuil_miles  part_trafic_pct  part_vols_retard_pct  part_minutes_pct
        1000            28.42                 29.13             29.03
        1500            13.63                 13.46             13.07
        2000             6.47                  6.40              6.28

Au seuil de 1500 miles : 13.5 % des vols en retard et 13.1 % des minutes de retard viennent des vols longs, pour 13.6 % du trafic.


**Réponse au sens littéral de l'hypothèse : réfutée, et largement.** Les vols longs ne portent
qu'environ un retard sur sept — très loin d'une « grande majorité ». Le résultat ne dépend pas du seuil :
même en appelant « long » tout vol de 1000 miles et plus, on plafonne autour d'un retard sur trois-quatre.

La clé de lecture est la comparaison avec la part de trafic : la part des vols longs dans les retards est
**quasi identique à leur part du ciel**. S'ils ne composent pas la majorité des retards, ce n'est pas
qu'ils soient plus fiables : c'est qu'ils sont minoritaires, et que le risque de retard ne varie presque
pas avec la distance — ce que la section suivante détaille.

### 7.2 Le mécanisme : ce que la distance fait vraiment au retard

Si les vols longs pesaient dans les retards plus que leur part de trafic, on le verrait ici : on regarde,
vol par vol, ce que la distance fait au retard d'arrivée.

In [22]:
BINS = [0, 500, 1000, 1500, 2000, 10_000]
BIN_LABELS = ["< 500 mi", "500 - 999 mi", "1000 - 1499 mi", "1500 - 1999 mi", ">= 2000 mi"]
distance_bin = pd.cut(completed_flights.DISTANCE, bins=BINS, labels=BIN_LABELS, right=False)

h2_distance = completed_flights.groupby(distance_bin, observed=True).agg(
    vols=("DISTANCE", "size"),
    taux_retard=("EST_EN_RETARD", "mean"),
    retard_moyen=("ARRIVAL_DELAY", "mean"),
    retard_depart_moyen=("DEPARTURE_DELAY", "mean"),
).reset_index()
h2_distance.columns = ["tranche", "vols", "taux_retard", "retard_moyen", "retard_depart_moyen"]
h2_distance["taux_retard"] *= 100
h2_distance["rattrapage_min"] = h2_distance.retard_depart_moyen - h2_distance.retard_moyen
h2_distance.round(2).to_string(index=False)

'       tranche    vols  taux_retard  retard_moyen  retard_depart_moyen  rattrapage_min\n      < 500 mi 2073950        18.21          5.16                 8.05            2.90\n  500 - 999 mi 2015991        18.65          4.64                 9.53            4.89\n1000 - 1499 mi  845177        19.72          4.49                10.89            6.40\n1500 - 1999 mi  409263        18.34          2.29                10.50            8.21\n    >= 2000 mi  369627        18.42          1.07                10.00            8.93'

In [23]:
shortest_bin = h2_distance.iloc[0]
longest_bin = h2_distance.iloc[-1]
print(f"Taux de retard  : {shortest_bin.tranche} = {shortest_bin.taux_retard:.2f} %  vs  "
      f"{longest_bin.tranche} = {longest_bin.taux_retard:.2f} %  ({longest_bin.taux_retard - shortest_bin.taux_retard:+.2f} pt)")
print(f"Retard moyen    : {shortest_bin.retard_moyen:.2f} min  vs  {longest_bin.retard_moyen:.2f} min "
      f"({longest_bin.retard_moyen - shortest_bin.retard_moyen:+.2f} min)")
print(f"Minutes rattrapées en vol : {shortest_bin.rattrapage_min:.2f} min  vs  {longest_bin.rattrapage_min:.2f} min")

Taux de retard  : < 500 mi = 18.21 %  vs  >= 2000 mi = 18.42 %  (+0.22 pt)
Retard moyen    : 5.16 min  vs  1.07 min (-4.09 min)
Minutes rattrapées en vol : 2.90 min  vs  8.93 min


**Le mécanisme apparaît ici.** Le taux de retard est quasi plat d'une tranche de distance à l'autre, mais le
retard **moyen** chute fortement quand la distance augmente. La différence entre retard au départ et retard
à l'arrivée montre pourquoi : plus un vol est long, plus il **rattrape** de minutes en vol, parce que les
horaires publiés intègrent une marge proportionnelle au temps de vol.

Autrement dit, la distance n'aggrave pas le retard à l'arrivée : elle donne au contraire de la marge pour
l'absorber. C'est pourquoi la part des vols longs dans les retards ne dépasse jamais leur part de trafic : la distance ne crée pas de sur-risque.

### 7.3 Contre-analyse : les compagnies spécialisées en vols longs

Reste une maniere indirecte de sauver l'hypothèse : même minoritaires, les vols longs pourraient dégrader
la ponctualité des compagnies qui en opèrent beaucoup. Point de méthode : cette question porte sur des
**compagnies**, pas sur des vols. L'unité d'analyse est donc la compagnie — il y en a 14. La tester sur
5,7 millions de vols reviendrait à compter chaque vol comme une observation indépendante alors que tous
les vols d'une même compagnie partagent son réseau, sa flotte et ses regles d'exploitation. C'est de la
**pseudo-réplication**, et ca fabrique de la significativité artificielle.

Test sur les 14 compagnies : corrélation de Pearson (relation linéaire), de Spearman (relation monotone,
robuste aux valeurs extrêmes) et régression linéaire pour chiffrer la pente.

In [24]:
airline_names = dict(zip(airlines_ref.IATA_CODE, airlines_ref.AIRLINE))

h2_airlines = completed_flights.groupby("AIRLINE", observed=True).agg(
    vols=("DISTANCE", "size"),
    vols_longs=("EST_VOL_LONG", "sum"),
    part_vols_longs=("EST_VOL_LONG", "mean"),
    taux_retard=("EST_EN_RETARD", "mean"),
    retard_moyen=("ARRIVAL_DELAY", "mean"),
    distance_moyenne=("DISTANCE", "mean"),
).reset_index()
h2_airlines["compagnie"] = h2_airlines.AIRLINE.map(airline_names)
h2_airlines["part_vols_longs"] *= 100
h2_airlines["taux_retard"] *= 100
h2_airlines = h2_airlines.sort_values("part_vols_longs", ascending=False).reset_index(drop=True)

h2_airlines[["AIRLINE", "compagnie", "vols", "part_vols_longs",
               "taux_retard", "retard_moyen", "distance_moyenne"]].round(2)

,AIRLINE,compagnie,vols,part_vols_longs,taux_retard,retard_moyen,distance_moyenne
0,VX,Virgin America,61248,43.20,19.23,4.74,1404.38
1,UA,United Air Lines Inc.,507762,33.92,20.62,5.43,1271.68
2,AS,Alaska Airlines Inc.,171439,27.76,13.04,-0.98,1198.69
3,B6,JetBlue Airways,262042,20.84,22.58,6.68,1063.05
4,HA,Hawaiian Airlines Inc.,76041,19.59,11.33,2.02,632.03
5,US,US Airways Inc.,194223,19.52,18.82,3.71,915.38
6,AA,American Airlines Inc.,712935,19.33,18.27,3.45,1042.37
7,DL,Delta Air Lines Inc.,870275,16.98,13.56,0.19,853.60
8,NK,Spirit Air Lines,115193,14.37,29.71,14.47,985.78
9,F9,Frontier Airlines Inc.,90090,13.85,26.16,12.50,967.16


In [25]:
x = h2_airlines.part_vols_longs.to_numpy()
y = h2_airlines.taux_retard.to_numpy()

r_p, p_p = stats.pearsonr(x, y)
r_s, p_s = stats.spearmanr(x, y)
reg = stats.linregress(x, y)

H2_CORR = {
    "n_compagnies": len(h2_airlines),
    "pearson_r": round(float(r_p), 3), "pearson_p": round(float(p_p), 3),
    "spearman_rho": round(float(r_s), 3), "spearman_p": round(float(p_s), 3),
    "pente": round(float(reg.slope), 4), "r2": round(float(reg.rvalue ** 2), 3),
    "regression_p": round(float(reg.pvalue), 3),
}

print(f"n = {H2_CORR['n_compagnies']} compagnies")
print(f"Pearson  r   = {H2_CORR['pearson_r']:+.3f}  (p = {H2_CORR['pearson_p']:.3f})")
print(f"Spearman rho = {H2_CORR['spearman_rho']:+.3f}  (p = {H2_CORR['spearman_p']:.3f})")
print(f"Régression   : {H2_CORR['pente']:+.4f} point de retard par point de vols longs, "
      f"R2 = {H2_CORR['r2']:.3f}, p = {H2_CORR['regression_p']:.3f}")
print()
print(f"La part de vols longs explique {H2_CORR['r2'] * 100:.1f} % de la variance du taux de retard "
      f"entre compagnies. Le signe est {'negatif' if r_p < 0 else 'positif'}, "
      f"soit {'l inverse de' if r_p < 0 else 'le sens de'} l'hypothèse.")

n = 14 compagnies
Pearson  r   = -0.179  (p = 0.539)
Spearman rho = -0.222  (p = 0.445)
Régression   : -0.0700 point de retard par point de vols longs, R2 = 0.032, p = 0.539

La part de vols longs explique 3.2 % de la variance du taux de retard entre compagnies. Le signe est negatif, soit l inverse de l'hypothèse.


### 7.4 Comparaison des deux groupes de compagnies

On coupe malgré tout les 14 compagnies à la médiane de part de vols longs, pour montrer explicitement ce
que produit — et ce que vaut — la comparaison qu'on aurait pu faire d'emblee.

In [26]:
MEDIAN_LONG_SHARE = float(h2_airlines.part_vols_longs.median())
high_share_airlines = set(h2_airlines.loc[h2_airlines.part_vols_longs >= MEDIAN_LONG_SHARE, "AIRLINE"])
flights["GROUPE_LONG"] = flights.AIRLINE.isin(high_share_airlines)

h2_groups = completed_flights.groupby(flights.GROUPE_LONG, observed=True).agg(
    vols=("EST_EN_RETARD", "size"), taux_retard=("EST_EN_RETARD", "mean")).reset_index()
h2_groups["segment"] = np.where(h2_groups.GROUPE_LONG,
                                 "Part forte de vols longs", "Part faible de vols longs")
h2_groups["taux_retard"] *= 100

HIGH_SHARE_RATE = float(h2_groups.loc[h2_groups.GROUPE_LONG, "taux_retard"].iloc[0])
LOW_SHARE_RATE = float(h2_groups.loc[~h2_groups.GROUPE_LONG, "taux_retard"].iloc[0])
H2_TEST = chi2_test(flights.GROUPE_LONG, flights.EST_EN_RETARD, completed_flights)

print(f"Médiane de part de vols longs : {MEDIAN_LONG_SHARE:.2f} %")
print(f"Part forte  : {HIGH_SHARE_RATE:.2f} % de retards")
print(f"Part faible : {LOW_SHARE_RATE:.2f} % de retards")
print(f"Écart       : {HIGH_SHARE_RATE - LOW_SHARE_RATE:+.2f} point")
print()
print(f"chi2 = {H2_TEST['chi2']:,.1f}  p = {H2_TEST['p_value']:.3g}  V de Cramer = {H2_TEST['cramer_v']}")
print(f"-> p très significative pour un écart de {HIGH_SHARE_RATE - LOW_SHARE_RATE:.2f} point et un V de "
      f"{H2_TEST['cramer_v']} : c'est exactement l'artefact de taille d'échantillon annonce plus haut.")
print(f"   Le test valide au niveau compagnie (n = 14) donne, lui, p = {H2_CORR['spearman_p']:.3f}.")

Médiane de part de vols longs : 18.16 %
Part forte  : 18.81 % de retards
Part faible : 18.51 % de retards
Écart       : +0.30 point

chi2 = 78.1  p = 1e-18  V de Cramer = 0.0037
-> p très significative pour un écart de 0.30 point et un V de 0.0037 : c'est exactement l'artefact de taille d'échantillon annonce plus haut.
   Le test valide au niveau compagnie (n = 14) donne, lui, p = 0.445.


### 7.5 Conclusion H2

In [27]:
H2_STATUS = "Réfutée"
H2_SUMMARY = (
    f"Les vols longs (>= {LONG_HAUL_THRESHOLD} miles) ne composent pas la majorité des retards : ils portent "
    f"{H2_COMPO['part_vols_retard_pct']:.1f} % des vols en retard et {H2_COMPO['part_minutes_pct']:.1f} % "
    f"des minutes de retard, pour {H2_COMPO['part_trafic_pct']:.1f} % des vols effectués. Leur part des "
    f"retards est donc exactement leur part du trafic : le taux de retard varie à peine avec la distance "
    f"({float(h2_distance.iloc[-1].taux_retard):.1f} % au-delà de 2000 miles contre "
    f"{float(h2_distance.iloc[0].taux_retard):.1f} % en dessous de 500), et les vols les plus longs "
    f"rattrapent en moyenne {float(h2_distance.iloc[-1].rattrapage_min):.1f} minutes en vol. La "
    f"contre-analyse par compagnie écarte aussi la version indirecte de l'hypothèse : sur les "
    f"{H2_CORR['n_compagnies']} compagnies, la corrélation entre part de vols longs et taux de retard est "
    f"de {H2_CORR['pearson_r']:+.3f} (Pearson, p = {H2_CORR['pearson_p']:.2f}), non significative et de "
    f"signe contraire à l'hypothèse."
)
print(H2_STATUS.upper()); print(); print(H2_SUMMARY)

RÉFUTÉE

Les vols longs (>= 1500 miles) ne composent pas la majorité des retards : ils portent 13.5 % des vols en retard et 13.1 % des minutes de retard, pour 13.6 % des vols effectués. Leur part des retards est donc exactement leur part du trafic : le taux de retard varie à peine avec la distance (18.4 % au-delà de 2000 miles contre 18.2 % en dessous de 500), et les vols les plus longs rattrapent en moyenne 8.9 minutes en vol. La contre-analyse par compagnie écarte aussi la version indirecte de l'hypothèse : sur les 14 compagnies, la corrélation entre part de vols longs et taux de retard est de -0.179 (Pearson, p = 0.54), non significative et de signe contraire à l'hypothèse.


**Réponse à l'hypothèse : réfutée.**

Les vols longs ne portent qu'environ un retard sur sept — très loin de la « grande majorité » annoncée.
Et ce n'est pas parce qu'ils seraient plus fiables : leur part des retards est identique à leur part du
trafic. L'hypothèse échoue parce que les vols longs sont **minoritaires dans le ciel domestique
américain**, et que la distance ne change presque rien au risque de retard — les horaires des vols longs
intègrent des marges qui absorbent en vol le retard pris au départ.

Ce qui sépare vraiment les compagnies ponctuelles des retardataires se lit dans le nuage de points :
Spirit et Frontier, low-cost à rotation tendue, sont les plus en retard malgré peu de vols longs ;
Delta, Alaska et Hawaiian sont les plus ponctuelles avec des profils de réseau opposés. La variable
explicative est le **modèle d'exploitation**, pas la distance.

**Ce que ca implique pour le dashboard** : répondre d'abord à la question posée — la part des retards
portée par les vols longs, comparée à leur part de trafic — puis montrer le nuage des 14 compagnies en
approfondissement.

---
## 8. H3 — « Une petite minorité de liaisons concentre la majorité des annulations »

Une route est un couple orienté `ORIGINE -> DESTINATION`. Périmètre : les onze mois en codes IATA
(cf. section 3.2).

Le piège de cette hypothèse est qu'elle est **presque vraie par construction** : le trafic aérien est lui-même
très concentré, donc les routes qui portent le plus d'annulations sont d'abord celles qui portent le plus de
vols. Pour que le résultat ait un sens, il faut comparer la concentration des annulations à **la concentration
du trafic**, et regarder les **taux** et pas seulement les volumes.

In [28]:
route_flights = flights[flights.ROUTE_LISIBLE]

routes = route_flights.groupby("ROUTE", observed=True).agg(
    vols=("CANCELLED", "size"),
    annulations=("CANCELLED", "sum"),
    distance=("DISTANCE", "mean"),
).reset_index()
routes["taux_annulation"] = routes.annulations / routes.vols * 100
routes[["origine", "destination"]] = routes.ROUTE.str.split(" -> ", expand=True)
routes = routes.sort_values(["annulations", "vols"], ascending=False).reset_index(drop=True)
routes["rang"] = np.arange(1, len(routes) + 1)
routes["part_routes_cumulee"] = routes.rang / len(routes) * 100
routes["part_annulations_cumulee"] = routes.annulations.cumsum() / routes.annulations.sum() * 100

N_ROUTES = len(routes)
N_CANCELLED = int(routes.annulations.sum())
print(f"Routes distinctes  : {N_ROUTES:,}")
print(f"Annulations        : {N_CANCELLED:,}  ({N_CANCELLED / int(flights.CANCELLED.sum()) * 100:.1f} % du total annuel)")
print(f"Vols du périmètre  : {int(routes.vols.sum()):,}")
routes.head(10)[["ROUTE", "vols", "annulations", "taux_annulation", "part_annulations_cumulee"]].round(2)

Routes distinctes  : 4,693
Annulations        : 87,430  (97.3 % du total annuel)
Vols du périmètre  : 5,332,914


,ROUTE,vols,annulations,taux_annulation,part_annulations_cumulee
0,BOS -> LGA,7096,443,6.24,0.51
1,LGA -> BOS,7100,441,6.21,1.01
2,LGA -> ORD,9639,435,4.51,1.51
3,ORD -> LGA,9575,408,4.26,1.98
4,LAX -> SFO,13457,342,2.54,2.37
5,LGA -> DCA,4303,340,7.90,2.76
6,SFO -> LAX,13744,338,2.46,3.14
7,DCA -> LGA,4303,334,7.76,3.52
8,DCA -> BOS,7686,271,3.53,3.83
9,BOS -> DCA,7687,266,3.46,4.14


### 8.1 Concentration des annulations, comparée à celle du trafic

In [29]:
def lorenz_curve(values_arr):
    """Part cumulée de la grandeur, triée du plus gros au plus petit."""
    v = np.sort(np.asarray(values_arr, dtype=float))[::-1]
    return np.cumsum(v) / v.sum() * 100

def gini(values_arr):
    """Indice de Gini : 0 = tout le monde pareil, 1 = tout concentre sur un seul."""
    v = np.sort(np.asarray(values_arr, dtype=float))
    n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum()))

cum_cancels = lorenz_curve(routes.annulations)
cum_traffic = lorenz_curve(routes.vols)
routes_share = np.arange(1, N_ROUTES + 1) / N_ROUTES * 100

idx_50 = int(np.argmax(cum_cancels >= 50))
N_ROUTES_50 = idx_50 + 1
PCT_ROUTES_50 = N_ROUTES_50 / N_ROUTES * 100

H3_CONC = {
    "routes": N_ROUTES, "annulations": N_CANCELLED,
    "routes_pour_50pct": N_ROUTES_50, "pct_routes_pour_50pct": round(PCT_ROUTES_50, 2),
    "gini_annulations": round(gini(routes.annulations), 3),
    "gini_vols": round(gini(routes.vols), 3),
}
for threshold in (10, 20, 30):
    k = int(np.ceil(N_ROUTES * threshold / 100)) - 1
    H3_CONC[f"annulations_top{threshold}pct"] = round(float(cum_cancels[k]), 2)
    H3_CONC[f"vols_top{threshold}pct"] = round(float(cum_traffic[k]), 2)

print(f"{N_ROUTES_50:,} routes ({PCT_ROUTES_50:.2f} %) portent 50 % des annulations.")
print()
print(f"{'seuil':>8} | {'% annulations':>14} | {'% des vols':>11} | {'ecart':>7}")
print("-" * 50)
for threshold in (10, 20, 30):
    a, v = H3_CONC[f"annulations_top{threshold}pct"], H3_CONC[f"vols_top{threshold}pct"]
    print(f"top {threshold:>2} % | {a:>13.2f} % | {v:>10.2f} % | {a - v:>+6.2f}")
print()
print(f"Gini des annulations : {H3_CONC['gini_annulations']:.3f}")
print(f"Gini du trafic       : {H3_CONC['gini_vols']:.3f}")

455 routes (9.70 %) portent 50 % des annulations.

   seuil |  % annulations |  % des vols |   ecart
--------------------------------------------------
top 10 % |         50.89 % |      38.72 % | +12.17
top 20 % |         70.42 % |      58.08 % | +12.34
top 30 % |         81.94 % |      71.13 % | +10.81

Gini des annulations : 0.683
Gini du trafic       : 0.558


**Lecture indispensable.** Énoncé seul, « 10 % des routes portent 51 % des annulations » impressionne. Mais
ces mêmes 10 % de routes portent déjà **39 % des vols**. L'essentiel de la concentration est donc mécanique :
il reflète la concentration du trafic sur les grands axes, pas une fragilité propre a ces routes.

L'information réelle est dans l'**écart entre les deux courbes**, confirme par les Gini (0,68 pour les
annulations contre 0,56 pour le trafic). Il existe bien un sur-risque, mais il est plus modeste que le chiffre
brut ne le laisse croire. Pour le mesurer proprement, on passe aux taux.

### 8.2 Le vrai signal : le taux d'annulation, pas le volume

In [30]:
priority_routes = routes.iloc[:N_ROUTES_50]
other_routes = routes.iloc[N_ROUTES_50:]

PRIORITY_RATE = priority_routes.annulations.sum() / priority_routes.vols.sum() * 100
OTHERS_RATE = other_routes.annulations.sum() / other_routes.vols.sum() * 100
EXCESS_RISK = PRIORITY_RATE / OTHERS_RATE

H3_CONC.update({
    "taux_annulation_prioritaires": round(float(PRIORITY_RATE), 2),
    "taux_annulation_autres": round(float(OTHERS_RATE), 2),
    "surrisque": round(float(EXCESS_RISK), 2),
})

print(f"{N_ROUTES_50} routes prioritaires : {PRIORITY_RATE:.2f} % d'annulation "
      f"({int(priority_routes.annulations.sum()):,} annulations sur {int(priority_routes.vols.sum()):,} vols)")
print(f"{len(other_routes):,} autres routes      : {OTHERS_RATE:.2f} % d'annulation")
print(f"\nSur-risque : x{EXCESS_RISK:.2f}")
print("\nC'est ce chiffre qui résiste à l'objection du volume : à nombre de vols égal, une route")
print("prioritaire est annulée deux à trois fois plus souvent qu'une autre.")

455 routes prioritaires : 2.92 % d'annulation (43,736 annulations sur 1,500,205 vols)
4,238 autres routes      : 1.14 % d'annulation

Sur-risque : x2.56

C'est ce chiffre qui résiste à l'objection du volume : à nombre de vols égal, une route
prioritaire est annulée deux à trois fois plus souvent qu'une autre.


In [31]:
# Quand ces routes sont-elles annulées ? Et sont-elles concentrées géographiquement ?
priority_set = set(priority_routes.ROUTE)
priority_flights = route_flights[route_flights.ROUTE.isin(priority_set)]

by_month = priority_flights.groupby("MONTH", observed=True).CANCELLED.agg(["size", "sum"])
by_month["taux"] = by_month["sum"] / by_month["size"] * 100
WINTER_SHARE = float(by_month.loc[by_month.index.isin([1, 2, 3]), "sum"].sum() / by_month["sum"].sum() * 100)

airport_cancels = route_flights.groupby(route_flights.ROUTE.str[:3], observed=True).agg(
    vols=("CANCELLED", "size"), annulations=("CANCELLED", "sum")).reset_index()
airport_cancels.columns = ["aeroport", "vols", "annulations"]
airport_cancels["taux_annulation"] = airport_cancels.annulations / airport_cancels.vols * 100
top_airports = airport_cancels.nlargest(10, "annulations").reset_index(drop=True)

print(f"Part des annulations des routes prioritaires survenue en janvier-mars : {WINTER_SHARE:.1f} %")
print(f"(pour référence, janvier-mars = {route_flights.MONTH.isin([1, 2, 3]).mean() * 100:.1f} % des vols)")
print()
print("Aéroports d'origine les plus touchés :")
print(top_airports.round(2).to_string(index=False))

Part des annulations des routes prioritaires survenue en janvier-mars : 51.7 %
(pour référence, janvier-mars = 26.3 % des vols)

Aéroports d'origine les plus touchés :
aeroport   vols  annulations  taux_annulation
     ORD 285884         8548             2.99
     DFW 239551         6254             2.61
     LGA  99605         4531             4.55
     EWR 101772         3110             3.06
     BOS 107847         2654             2.46
     ATL 346836         2557             0.74
     LAX 194673         2164             1.11
     SFO 148008         2148             1.45
     IAH 146622         2130             1.45
     DEN 196055         2123             1.08


**Le phénomène a un nom.** Plus de la moitié des annulations des routes prioritaires tombent sur le premier
trimestre, alors que ce trimestre ne représente qu'un quart des vols. Et les aéroports en tête ne sont pas les
plus gros du pays — Atlanta, premier aéroport mondial en trafic, affiche un taux de 0,74 %, quand LaGuardia
est à 4,55 %.

H3 ne décrit donc pas « des routes fragiles » en général : elle décrit la **navette court-courrier du Nord-Est
en hiver** (New York, Boston, Washington, Chicago), où des rotations très fréquentes sur de courtes distances
rencontrent la saison des tempêtes. C'est cohérent avec le fait que la météo est le premier motif d'annulation
du dataset, et avec les pires journées identifiées en section 5.

In [32]:
# Table des routes à afficher, enrichie des coordonnées pour la carte du dashboard.
coords = (airports_ref.dropna(subset=["LATITUDE", "LONGITUDE"])
          .set_index("IATA_CODE")[["AIRPORT", "CITY", "STATE", "LATITUDE", "LONGITUDE"]])

routes_geo = routes.join(coords.add_prefix("o_"), on="origine").join(coords.add_prefix("d_"), on="destination")
missing_coords = routes_geo.o_LATITUDE.isna() | routes_geo.d_LATITUDE.isna()
print(f"Routes sans coordonnées complètes : {int(missing_coords.sum())} sur {N_ROUTES} "
      f"({routes_geo.loc[missing_coords, 'annulations'].sum() / N_CANCELLED * 100:.2f} % des annulations)")

TOP_ROUTES = routes.head(15)[["rang", "ROUTE", "origine", "destination", "vols",
                              "annulations", "taux_annulation", "distance"]].copy()
TOP_ROUTES["ville_origine"] = TOP_ROUTES.origine.map(coords.CITY)
TOP_ROUTES["ville_destination"] = TOP_ROUTES.destination.map(coords.CITY)
TOP_ROUTES.round(2)

Routes sans coordonnées complètes : 22 sur 4693 (0.07 % des annulations)


,rang,ROUTE,origine,destination,vols,annulations,taux_annulation,distance,ville_origine,ville_destination
0,1,BOS -> LGA,BOS,LGA,7096,443,6.24,184.0,Boston,New York
1,2,LGA -> BOS,LGA,BOS,7100,441,6.21,184.0,New York,Boston
2,3,LGA -> ORD,LGA,ORD,9639,435,4.51,733.0,New York,Chicago
3,4,ORD -> LGA,ORD,LGA,9575,408,4.26,733.0,Chicago,New York
4,5,LAX -> SFO,LAX,SFO,13457,342,2.54,337.0,Los Angeles,San Francisco
5,6,LGA -> DCA,LGA,DCA,4303,340,7.90,214.0,New York,Arlington
6,7,SFO -> LAX,SFO,LAX,13744,338,2.46,337.0,San Francisco,Los Angeles
7,8,DCA -> LGA,DCA,LGA,4303,334,7.76,214.0,Arlington,New York
8,9,DCA -> BOS,DCA,BOS,7686,271,3.53,399.0,Arlington,Boston
9,10,BOS -> DCA,BOS,DCA,7687,266,3.46,399.0,Boston,Arlington


In [33]:
# Routes au taux d'annulation le plus élevé : on impose un volume minimum, sinon une route
# a 4 vols dont 1 annulé sortirait en tête avec 25 %.
MIN_VOLUME = 365     # environ un vol par jour sur l'année
high_rate_routes = (routes[routes.vols >= MIN_VOLUME]
                 .nlargest(12, "taux_annulation")
                 .reset_index(drop=True))
print(f"Routes >= {MIN_VOLUME} vols retenues : {int((routes.vols >= MIN_VOLUME).sum()):,} sur {N_ROUTES:,}")
high_rate_routes[["ROUTE", "vols", "annulations", "taux_annulation"]].round(2)

Routes >= 365 vols retenues : 3,136 sur 4,693


,ROUTE,vols,annulations,taux_annulation
0,ORF -> LGA,402,61,15.17
1,LGA -> ORF,515,68,13.20
2,DCA -> JFK,996,114,11.45
3,JFK -> DCA,1002,106,10.58
4,RDU -> LGA,1878,197,10.49
5,LGA -> BHM,490,51,10.41
6,CHO -> LGA,396,41,10.35
7,LGA -> GSO,1295,134,10.35
8,LGA -> DAY,403,41,10.17
9,GSO -> LGA,1295,129,9.96


### 8.3 Robustesse du résultat

In [34]:
# 1) Le résultat dépend-il du périmètre IATA (= de l'exclusion d'octobre) ?
routes_no_october = (route_flights[route_flights.MONTH.ne(10)]
                   .groupby("ROUTE", observed=True).CANCELLED.agg(["size", "sum"]))
c = np.sort(routes_no_october["sum"].to_numpy())[::-1]
cum = np.cumsum(c) / c.sum() * 100
pct_no_october = (int(np.argmax(cum >= 50)) + 1) / len(c) * 100

# 2) Le résultat dépend-il du choix de compter les routes orientées (A->B et B->A séparément) ?
pair_flights = route_flights.assign(
    PAIRE=[" <-> ".join(sorted(p)) for p in zip(route_flights.ROUTE.str[:3], route_flights.ROUTE.str[-3:])])
pair_cancels = pair_flights.groupby("PAIRE", observed=True).CANCELLED.sum()
c2 = np.sort(pair_cancels.to_numpy())[::-1]
cum2 = np.cumsum(c2) / c2.sum() * 100
pct_pairs = (int(np.argmax(cum2 >= 50)) + 1) / len(c2) * 100

print(f"Référence (routes orientées, 11 mois)   : 50 % des annulations dans {PCT_ROUTES_50:.2f} % des routes")
print(f"Variante  (octobre exclu explicitement) : {pct_no_october:.2f} %  -> identique par construction,")
print(f"                                          le filtre IATA équivaut exactement à retirer octobre")
print(f"Variante  (liaisons non orientées)      : {pct_pairs:.2f} % des {len(c2):,} liaisons")
print(f"\nLe résultat ne dépend pas de ces choix de périmètre.")

Référence (routes orientées, 11 mois)   : 50 % des annulations dans 9.70 % des routes
Variante  (octobre exclu explicitement) : 9.70 %  -> identique par construction,
                                          le filtre IATA équivaut exactement à retirer octobre
Variante  (liaisons non orientées)      : 9.62 % des 2,381 liaisons

Le résultat ne dépend pas de ces choix de périmètre.


### 8.4 Conclusion H3

In [35]:
H3_STATUS = "Confirmée, avec une réserve importante"
H3_SUMMARY = (
    f"{fmt_fr(N_ROUTES_50)} routes sur {fmt_fr(N_ROUTES)} ({PCT_ROUTES_50:.2f} %) portent la moitié des "
    f"{fmt_fr(N_CANCELLED)} annulations du périmètre, et le Gini des annulations ({H3_CONC['gini_annulations']:.3f}) "
    f"dépasse celui du trafic ({H3_CONC['gini_vols']:.3f}). La concentration est donc réelle, mais elle est "
    f"pour une bonne part mécanique : les 10 % de routes en tête portent "
    f"{H3_CONC['annulations_top10pct']:.1f} % des annulations et déjà {H3_CONC['vols_top10pct']:.1f} % des vols. "
    f"Le résultat solide est le sur-risque à volume comparable : {PRIORITY_RATE:.2f} % d'annulation sur les routes "
    f"prioritaires contre {OTHERS_RATE:.2f} % ailleurs, soit x{EXCESS_RISK:.2f}. Ces routes sont majoritairement "
    f"des navettes courtes du Nord-Est, et {WINTER_SHARE:.0f} % de leurs annulations tombent au premier trimestre."
)
print(H3_STATUS.upper()); print(); print(H3_SUMMARY)

CONFIRMÉE, AVEC UNE RÉSERVE IMPORTANTE

455 routes sur 4 693 (9.70 %) portent la moitié des 87 430 annulations du périmètre, et le Gini des annulations (0.683) dépasse celui du trafic (0.558). La concentration est donc réelle, mais elle est pour une bonne part mécanique : les 10 % de routes en tête portent 50.9 % des annulations et déjà 38.7 % des vols. Le résultat solide est le sur-risque à volume comparable : 2.92 % d'annulation sur les routes prioritaires contre 1.14 % ailleurs, soit x2.56. Ces routes sont majoritairement des navettes courtes du Nord-Est, et 52 % de leurs annulations tombent au premier trimestre.


**Réponse à l'hypothèse : confirmée, mais le chiffre spectaculaire doit être relativise.**

Oui, une minorité de routes concentre la majorité des annulations. Mais présenter « 9,7 % des routes = 50 %
des annulations » sans préciser que ces routes portent déjà une part énorme du trafic serait un raccourci
trompeur. Le résultat défendable est le **sur-risque à volume comparable**, et son explication : un effet
conjoint de réseau court-courrier dense et de météo hivernale sur le Nord-Est.

**Ce que ca implique pour le dashboard** : la carte seule ne démontre rien. Il faut afficher les deux courbes
de concentration superposées — l'écart entre elles *est* la démonstration.

---
## 9. Figures

Les mêmes graphiques que le dashboard, en PNG, pour le rapport écrit et le support de presentation.
Ils sont enregistrés dans `v2/outputs/figures/`.

In [36]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9d3d7", "axes.labelcolor": "#4a5c63",
    "text.color": "#172126", "xtick.color": "#4a5c63", "ytick.color": "#4a5c63",
    "grid.color": "#e3e9eb",
})
TEAL, ORANGE, GRIS, BLEU, JAUNE = "#087f8c", "#e45c3a", "#94a3b8", "#3977a8", "#d69e2e"

def save_fig(fig, name):
    out_path = FIG / f"{name}.png"
    fig.savefig(out_path)
    plt.close(fig)
    print("figure ->", out_path.relative_to(PROJECT_ROOT))

In [37]:
# Figure 1 - H1 : le rythme des retards jour par jour
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(daily.DATE, daily.taux_retard, color=TEAL, lw=1.1)
for name, (mask, _) in HOLIDAY_WINDOWS.items():
    days = flights.loc[mask, "DATE"]
    ax.axvspan(days.min(), days.max(), color=ORANGE, alpha=0.10, lw=0)
ax.axhline(GLOBAL_DELAY_RATE, color=GRIS, ls="--", lw=1)
ax.annotate(f"moyenne annuelle {GLOBAL_DELAY_RATE:.1f} %",
            (daily.DATE.iloc[262], GLOBAL_DELAY_RATE), textcoords="offset points",
            xytext=(0, -14), color="#4a5c63", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=.85))
for _, r in WORST_DELAY_DAYS.head(3).iterrows():
    ax.annotate(r.DATE.strftime("%d %b"), (r.DATE, r.taux_retard), textcoords="offset points",
                xytext=(0, 7), ha="center", fontsize=8, color=ORANGE, fontweight="bold")
ax.set_title("H1 - Taux de vols en retard, jour par jour (zones orangées : fenêtres de forte mobilité)")
ax.set_ylabel("% de vols en retard"); ax.grid(axis="y", alpha=.6)
save_fig(fig, "h1_rythme_journalier")

figure -> v2/outputs/figures/h1_rythme_journalier.png


In [38]:
# Figure 2 - H1 : effet net de chaque fenêtre, une fois la référence neutralisée
fig, ax = plt.subplots(figsize=(7.5, 3.6))
d_ = h1_controlled.sort_values("effet_net_points")
colors = [ORANGE if v > 0.5 else (BLEU if v < -0.5 else GRIS) for v in d_.effet_net_points]
bars = ax.barh(d_.fenetre, d_.effet_net_points, color=colors, height=.62)
ax.axvline(0, color="#4a5c63", lw=1.2)
for b, v in zip(bars, d_.effet_net_points):
    ax.text(v + (0.5 if v > 0 else -0.5), b.get_y() + b.get_height() / 2, f"{v:+.1f} pt",
            va="center", ha="left" if v > 0 else "right", fontsize=9, fontweight="bold")
ax.set_xlim(min(d_.effet_net_points) - 5, max(d_.effet_net_points) + 5)
ax.set_title("H1 - Effet net sur le taux de retard, vs période de référence comparable")
ax.set_xlabel("écart en points de %"); ax.grid(axis="x", alpha=.6)
save_fig(fig, "h1_effet_net_fenetres")

figure -> v2/outputs/figures/h1_effet_net_fenetres.png


In [39]:
# Figure 3 - H2 : absence de relation entre part de vols longs et retard
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.scatter(h2_airlines.part_vols_longs, h2_airlines.taux_retard,
           s=h2_airlines.vols / 4000, color=TEAL, alpha=.75, edgecolor="white", zorder=3)
xs = np.linspace(0, h2_airlines.part_vols_longs.max() * 1.08, 50)
ax.plot(xs, reg.intercept + reg.slope * xs, color=ORANGE, ls="--", lw=1.6, zorder=2)
for _, r in h2_airlines.iterrows():
    ax.annotate(r.AIRLINE, (r.part_vols_longs, r.taux_retard), textcoords="offset points",
                xytext=(7, 4), fontsize=8.5, fontweight="bold", color="#172126")
ax.text(.98, .96, f"r = {H2_CORR['pearson_r']:+.3f}   p = {H2_CORR['pearson_p']:.2f}\n"
                  f"R2 = {H2_CORR['r2']:.3f}   n = {H2_CORR['n_compagnies']}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#4a5c63",
        bbox=dict(boxstyle="round,pad=0.45", fc="#f4f7f8", ec="#dbe3e6"))
ax.set_title("H2 - Aucune relation entre part de vols longs et ponctualité")
ax.set_xlabel("part de vols >= 1500 miles (%)"); ax.set_ylabel("% de vols en retard")
ax.grid(alpha=.6)
save_fig(fig, "h2_nuage_compagnies")

figure -> v2/outputs/figures/h2_nuage_compagnies.png


In [40]:
# Figure 4 - H2 : ce que fait vraiment la distance
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.bar(h2_distance.tranche.astype(str), h2_distance.taux_retard, color=TEAL, width=.62)
a1.axhline(GLOBAL_DELAY_RATE, color=GRIS, ls="--", lw=1)
a1.set_title("Taux de retard : quasi plat"); a1.set_ylabel("% de vols en retard")
a1.set_ylim(0, 26); a1.grid(axis="y", alpha=.6)
a2.bar(h2_distance.tranche.astype(str), h2_distance.rattrapage_min, color=ORANGE, width=.62)
a2.set_title("Minutes rattrapées en vol : croissantes"); a2.set_ylabel("minutes")
a2.grid(axis="y", alpha=.6)
for a in (a1, a2):
    a.tick_params(axis="x", rotation=20, labelsize=8.5)
fig.suptitle("H2 - La distance ne dégrade pas l'arrivée, elle donne de la marge pour absorber le retard",
             fontsize=11, fontweight="bold", y=1.04)
save_fig(fig, "h2_effet_distance")

figure -> v2/outputs/figures/h2_effet_distance.png


In [41]:
# Figure 5 - H3 : concentration des annulations vs concentration du trafic
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.fill_between(routes_share, cum_traffic, cum_cancels, color=ORANGE, alpha=.13,
                label="sur-concentration des annulations")
ax.plot(routes_share, cum_cancels, color=ORANGE, lw=2.4, label="annulations cumulées")
ax.plot(routes_share, cum_traffic, color=BLEU, lw=2.0, ls="-", label="vols cumulés")
ax.plot([0, 100], [0, 100], color=GRIS, ls=":", lw=1.2, label="répartition uniforme")
ax.axhline(50, color=GRIS, ls="--", lw=.9)
ax.axvline(PCT_ROUTES_50, color=GRIS, ls="--", lw=.9)
ax.annotate(f"{PCT_ROUTES_50:.1f} % des routes\n= 50 % des annulations",
            (PCT_ROUTES_50, 50), textcoords="offset points", xytext=(14, -34), fontsize=9,
            color="#172126", arrowprops=dict(arrowstyle="->", color="#4a5c63", lw=.9))
ax.set_title("H3 - La concentration des annulations dépasse celle du trafic, mais de peu")
ax.set_xlabel("part cumulée des routes (%)"); ax.set_ylabel("part cumulée (%)")
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.grid(alpha=.6); ax.legend(fontsize=8.5, loc="lower right")
save_fig(fig, "h3_concentration_lorenz")

figure -> v2/outputs/figures/h3_concentration_lorenz.png


In [42]:
# Figure 6 - H3 : où et quand
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.0))
t = TOP_ROUTES.head(10).iloc[::-1]
a1.barh(t.ROUTE, t.annulations, color=ORANGE, height=.66)
for i, (n, tx) in enumerate(zip(t.annulations, t.taux_annulation)):
    a1.text(n + 6, i, f"{tx:.1f} %", va="center", fontsize=8.5, color="#4a5c63")
a1.set_title("Routes les plus annulées"); a1.set_xlabel("vols annulés"); a1.grid(axis="x", alpha=.6)
a2.bar(by_month.index, by_month["taux"], color=[ORANGE if m in (1, 2, 3) else TEAL for m in by_month.index], width=.66)
a2.set_title(f"Taux d'annulation des {N_ROUTES_50} routes prioritaires, par mois")
a2.set_xlabel("mois"); a2.set_ylabel("% annulés"); a2.set_xticks(range(1, 13)); a2.grid(axis="y", alpha=.6)
fig.suptitle("H3 - Une navette court-courrier du Nord-Est, concentrée sur l'hiver",
             fontsize=11, fontweight="bold", y=1.02)
save_fig(fig, "h3_routes_et_saison")

figure -> v2/outputs/figures/h3_routes_et_saison.png


---
## 10. Exports

Trois familles de livrables :

1. **CSV** — tableaux réutilisables dans Excel, Tableau ou le rapport écrit.
2. **`resume_v2.json`** — synthèse chiffrée de toutes les hypothèses.
3. **`dashboard/dashboard_data.js`** — les données du dashboard HTML.

Sur ce troisième point : le dashboard ne contient **aucun chiffre en dur**. Il lit ce fichier, qui est
régénéré ici. Réexécuter le notebook met donc le dashboard à jour automatiquement, et il est impossible
qu'il affiche autre chose que ce que le notebook a calculé.

Le format retenu est un `.js` (`window.DASHBOARD_DATA = {...}`) et non un `.json` : un navigateur refuse de
charger un JSON local via `fetch()` pour raison de sécurité (CORS sur `file://`), alors qu'une balise
`<script>` fonctionne. Le dashboard s'ouvre donc par un simple double-clic, sans serveur web.

In [43]:
exports_csv = {
    "kpi_panorama": pd.DataFrame([kpi]).T.rename(columns={0: "valeur"}),
    "serie_mensuelle": monthly,
    "serie_journaliere": daily,
    "causes_retard": delay_causes[["cause", "part_minutes_pct"]],
    "causes_annulation": cancel_causes[["cause", "vols", "part_pct"]],
    "h1_fenetres_detail": h1_detail,
    "h1_effet_net_controle": h1_controlled,
    "h2_compagnies": h2_airlines,
    "h2_tranches_distance": h2_distance,
    "h2_composition": h2_compo,
    "h3_routes_completes": routes,
    "h3_top_routes": TOP_ROUTES,
    "h3_routes_plus_risquees": high_rate_routes,
    "h3_top_aeroports": top_airports,
}
for name, table in exports_csv.items():
    out_path = OUT / f"{name}.csv"
    table.to_csv(out_path, index=(name == "kpi_panorama"), encoding="utf-8-sig")
    print(f"{out_path.name:34s} {len(table):>7,} lignes")

kpi_panorama.csv                        11 lignes
serie_mensuelle.csv                     12 lignes
serie_journaliere.csv                  365 lignes
causes_retard.csv                        5 lignes
causes_annulation.csv                    4 lignes
h1_fenetres_detail.csv                   5 lignes
h1_effet_net_controle.csv                4 lignes
h2_compagnies.csv                       14 lignes
h2_tranches_distance.csv                 5 lignes
h2_composition.csv                       3 lignes
h3_routes_completes.csv              4,693 lignes
h3_top_routes.csv                       15 lignes
h3_routes_plus_risquees.csv             12 lignes
h3_top_aeroports.csv                    10 lignes


In [44]:
SUMMARY = {
    "genere_le": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "source": "Kaggle / US DOT - 2015 Flight Delays and Cancellations (vols domestiques US)",
    "perimetre": "Année 2015 complète, 12 mois (H3 : 11 mois, cf. codes aéroport d'octobre)",
    "definitions": {
        "retard": f"arrivée avec >= {DELAY_THRESHOLD} min de retard, hors vols annulés et déroutés",
        "vol_long": f"distance >= {LONG_HAUL_THRESHOLD} miles (long-courrier domestique)",
        "vacances": {k: v for k, (_, v) in HOLIDAY_WINDOWS.items()},
        "route": "couple orienté ORIGINE -> DESTINATION en codes IATA",
    },
    "panorama": kpi,
    "H1": {
        "statut": H1_STATUS, "resume": H1_SUMMARY,
        "taux_vacances_pct": round(HOLIDAY_RATE, 2), "taux_hors_vacances_pct": round(BASELINE_RATE, 2),
        "ecart_brut_points": round(NAIVE_GAP, 2), "test_brut": H1_TEST,
        "poids_ete_dans_vacances_pct": round(SUMMER_WEIGHT, 1),
        "effet_net_par_fenetre": h1_controlled.round(3).to_dict("records"),
    },
    "H2": {
        "statut": H2_STATUS, "resume": H2_SUMMARY, "composition": H2_COMPO,
        "correlations": H2_CORR, "test_groupes": H2_TEST,
        "taux_part_forte_pct": round(HIGH_SHARE_RATE, 2), "taux_part_faible_pct": round(LOW_SHARE_RATE, 2),
        "ecart_points": round(HIGH_SHARE_RATE - LOW_SHARE_RATE, 2),
        "mediane_part_vols_longs_pct": round(MEDIAN_LONG_SHARE, 2),
    },
    "H3": {
        "statut": H3_STATUS, "resume": H3_SUMMARY, "concentration": H3_CONC,
        "part_annulations_janvier_mars_pct": round(WINTER_SHARE, 1),
        "route_la_plus_annulee": {
            "route": routes.iloc[0].ROUTE, "annulations": int(routes.iloc[0].annulations),
            "vols": int(routes.iloc[0].vols), "taux_pct": round(float(routes.iloc[0].taux_annulation), 2),
        },
    },
    "duree_execution_sec": None,
}
(OUT / "resume_v2.json").write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in SUMMARY.items() if k in ("H1", "H2", "H3")},
                 indent=2, ensure_ascii=False)[:1400], "...")

{
  "H1": {
    "statut": "Confirmée, mais à préciser",
    "resume": "L'écart brut est de +5.03 points (21.91 % contre 16.89 %), mais cet agrégat est dominé à 77 % par l'été et masque des comportements opposés. Une fois chaque fenêtre comparée à une référence comparable, l'effet le plus fort est Nouvel an (+18.25 points vs reste du même mois), l'été ne pèse que +4.12 points, et Thanksgiving est negatif (-0.82 point). Ce ne sont donc pas 'les vacances' qui créent du retard, mais les pics courts de fin d'année.",
    "taux_vacances_pct": 21.91,
    "taux_hors_vacances_pct": 16.89,
    "ecart_brut_points": 5.03,
    "test_brut": {
      "chi2": 21472.5,
      "p_value": 0.0,
      "dof": 1,
      "cramer_v": 0.0613,
      "risque_relatif": 1.298,
      "n": 5714008
    },
    "poids_ete_dans_vacances_pct": 77.1,
    "effet_net_par_fenetre": [
      {
        "fenetre": "Nouvel an",
        "nature": "effet fenêtre",
        "reference": "reste du même mois",
        "vols": 76931,
      

### 10.1 Carte : projection des routes en Python

Le dashboard doit s'ouvrir hors ligne, sans dépendre d'un CDN. On fait donc en Python tout ce qu'une
librairie JavaScript de cartographie ferait dans le navigateur : lire le fond de carte des États, le
**projeter en Albers Equal Area** (projection standard pour les États-Unis, qui préserve les surfaces),
et exporter directement des tracés SVG.

Le dashboard n'a alors plus qu'à afficher des chemins déjà calculés : zéro librairie externe, zéro appel réseau.

In [45]:
BASEMAP_FILE = ASSETS / "states-10m.json"
BASEMAP_URL = "https://cdn.jsdelivr.net/npm/us-atlas@3/states-10m.json"

if not BASEMAP_FILE.exists():
    import urllib.request
    print("Téléchargement du fond de carte ...")
    BASEMAP_FILE.write_bytes(urllib.request.urlopen(BASEMAP_URL, timeout=90).read())
topo = json.loads(BASEMAP_FILE.read_text(encoding="utf-8"))
print(f"Fond de carte : {BASEMAP_FILE.relative_to(PROJECT_ROOT)} ({BASEMAP_FILE.stat().st_size / 1024:.0f} Ko)")

Fond de carte : v2/assets/states-10m.json (112 Ko)


In [46]:
RAD = math.pi / 180

class AlbersConic:
    """Projection conique équivalente d'Albers (equal-area), paramétrée comme d3.geoConicEqualArea."""

    def __init__(self, parallels, meridian, center, scale, translation):
        s0 = math.sin(parallels[0] * RAD)
        self.n = (s0 + math.sin(parallels[1] * RAD)) / 2
        self.c = 1 + s0 * (2 * self.n - s0)
        self.r0 = math.sqrt(self.c) / self.n
        self.meridian, self.k = meridian, scale
        # Le centre est exprimé dans le repère DÉJÀ tourne : on ne lui reapplique pas le méridien.
        cx, cy = self._raw(center[0], center[1])
        self.dx = translation[0] - scale * cx
        self.dy = translation[1] + scale * cy

    def _raw(self, lon_tourne, lat):
        # Ramener la longitude dans [-180, 180] : sans cela les Aléoutiennes, qui franchissent
        # l'antimeridien (172 degrés Est), seraient projetées à l'oppose de l'Alaska.
        lon_tourne = (lon_tourne + 180) % 360 - 180
        r = math.sqrt(self.c - 2 * self.n * math.sin(lat * RAD)) / self.n
        t = lon_tourne * RAD * self.n
        return r * math.sin(t), self.r0 - r * math.cos(t)

    def __call__(self, lon, lat):
        x, y = self._raw(lon + self.meridian, lat)
        return self.k * x + self.dx, self.dy - self.k * y


MAP_W, MAP_H = 860, 470

def albers_usa(width, height):
    """Projection composite : 48 États contigus + encarts Alaska et Hawaii."""
    k, tx, ty = width * 1.12, width / 2, height / 2
    return {
        "48": AlbersConic([29.5, 45.5], 96, (-0.6, 38.7), k, (tx, ty)),
        "AK": AlbersConic([55, 65], 154, (-2, 58.5), k * 0.32, (tx - 0.30 * k, ty + 0.20 * k)),
        "HI": AlbersConic([8, 18], 157, (-3, 19.9), k, (tx - 0.205 * k, ty + 0.212 * k)),
    }

PROJ = albers_usa(MAP_W, MAP_H)
AK_HI_FIPS = {"02": "AK", "15": "HI"}
TERRITORIES = {"72", "78", "60", "66", "69"}     # Porto Rico, iles Vierges, Samoa, Guam, Mariannes

def project_point(state_fips, lon, lat):
    return PROJ[AK_HI_FIPS.get(state_fips, "48")](lon, lat)

def decode_arcs(topo):
    """TopoJSON : les arcs sont quantifiés et encodes en deltas. On les remet en lon/lat."""
    (sx, sy), (tx, ty) = topo["transform"]["scale"], topo["transform"]["translate"]
    arcs = []
    for arc in topo["arcs"]:
        x = y = 0
        points = []
        for dx, dy in arc:
            x += dx; y += dy
            points.append((x * sx + tx, y * sy + ty))
        arcs.append(points)
    return arcs

def assemble_ring(arcs, indices):
    """Un anneau est une suite d'arcs ; un indice négatif ~i signifie l'arc i parcouru à l'envers."""
    points = []
    for i in indices:
        segment = arcs[~i][::-1] if i < 0 else arcs[i]
        points.extend(segment if not points else segment[1:])
    return points

arcs = decode_arcs(topo)
TOLERANCE = 1.0      # décimation en pixels : allège le fichier sans effet visible

state_paths = []
for geo in topo["objects"]["states"]["geometries"]:
    if geo["id"] in TERRITORIES:
        continue
    rings = ([geo["arcs"]] if geo["type"] == "Polygon" else geo["arcs"])
    pieces = []
    for polygon in rings:
        for ring in polygon:
            pts = []
            for lon, lat in assemble_ring(arcs, ring):
                x, y = project_point(geo["id"], lon, lat)
                if not pts or abs(x - pts[-1][0]) + abs(y - pts[-1][1]) > TOLERANCE:
                    pts.append((x, y))
            if len(pts) >= 3:
                pieces.append("M" + "L".join(f"{x:.1f},{y:.1f}" for x, y in pts) + "Z")
    if pieces:
        state_paths.append("".join(pieces))

print(f"{len(state_paths)} États projetés, "
      f"{sum(len(t) for t in state_paths) / 1024:.0f} Ko de tracés SVG")

51 États projetés, 93 Ko de tracés SVG


In [47]:
# État de rattachement de chaque aéroport, pour choisir la bonne projection (encart AK / HI)
FIPS = {"AK": "02", "HI": "15"}
airport_state = airports_ref.set_index("IATA_CODE").STATE.to_dict()

def position(code_iata):
    if code_iata not in coords.index:
        return None
    lat, lon = coords.loc[code_iata, "LATITUDE"], coords.loc[code_iata, "LONGITUDE"]
    if pd.isna(lat) or pd.isna(lon):
        return None
    return project_point(FIPS.get(airport_state.get(code_iata), "48"), float(lon), float(lat))

def arc_svg(p1, p2):
    """Arc de cercle stylisé entre deux aéroports (courbe de Bézier quadratique)."""
    (x1, y1), (x2, y2) = p1, p2
    seg_length = math.hypot(x2 - x1, y2 - y1)
    curvature = min(70, max(14, seg_length * 0.19))
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2 - curvature
    return f"M{x1:.1f},{y1:.1f}Q{mx:.1f},{my:.1f} {x2:.1f},{y2:.1f}"

N_MAP_ROUTES = 180
# les routes tracées : le top 180 en volume d'annulations, PLUS les routes au plus fort taux
# (sinon la carte ne pourrait pas surligner les routes du classement « plus risquées »)
map_routes = pd.concat([routes.head(N_MAP_ROUTES), high_rate_routes]).drop_duplicates(subset="ROUTE")
route_paths, skipped = [], 0
for _, r in map_routes.iterrows():
    a, b = position(r.origine), position(r.destination)
    if a is None or b is None:
        skipped += 1
        continue
    route_paths.append({"route": r.ROUTE, "d": arc_svg(a, b),
                          "annulations": int(r.annulations), "vols": int(r.vols),
                          "taux": round(float(r.taux_annulation), 2), "rang": int(r.rang)})

shown_codes = sorted({c for r in routes.head(10).itertuples()
                         for c in (r.origine, r.destination)})
markers = []
for c in shown_codes:
    p = position(c)
    if p:
        match_row = airport_cancels[airport_cancels.aeroport.eq(c)]
        markers.append({"code": c, "x": round(p[0], 1), "y": round(p[1], 1),
                          "ville": coords.CITY.get(c, ""),
                          "annulations": int(match_row.annulations.iloc[0]) if len(match_row) else 0,
                          "taux": round(float(match_row.taux_annulation.iloc[0]), 2) if len(match_row) else 0.0})

print(f"{len(route_paths)} routes tracées ({skipped} ignorees faute de coordonnées), "
      f"{len(markers)} aéroports marqués")

187 routes tracées (0 ignorees faute de coordonnées), 6 aéroports marqués


In [48]:
def downsample(xs, ys, n=200):
    """Sous-echantillonne une courbe pour le dashboard, en gardant les extrémités."""
    idx = np.unique(np.linspace(0, len(xs) - 1, n).astype(int))
    return [[round(float(xs[i]), 3), round(float(ys[i]), 3)] for i in idx]

DASHBOARD_PAYLOAD = {
    "meta": {
        "genere_le": SUMMARY["genere_le"], "source": SUMMARY["source"],
        "perimetre": SUMMARY["perimetre"],
        "definitions": {"retard": SUMMARY["definitions"]["retard"],
                        "vol_long": SUMMARY["definitions"]["vol_long"],
                        "route": SUMMARY["definitions"]["route"]},
    },
    "kpi": kpi,
    "reperes": {"taux_retard_global": round(GLOBAL_DELAY_RATE, 2),
                "taux_annulation_global": round(GLOBAL_CANCEL_RATE, 2)},
    "mensuel": monthly.round(2).to_dict("records"),
    "journalier": [{"j": int(r.jour_annee), "date": r.DATE.strftime("%d/%m"),
                    "retard": round(float(r.taux_retard), 2),
                    "annulation": round(float(r.taux_annulation), 2)}
                   for r in daily.itertuples()],
    "fenetres": [{"nom": n, "debut": int(flights.loc[m, "JOUR_ANNEE"].min()),
                  "fin": int(flights.loc[m, "JOUR_ANNEE"].max()), "dates": lib}
                 for n, (m, lib) in HOLIDAY_WINDOWS.items()],
    "causes_retard": delay_causes[["cause", "part_minutes_pct"]].round(2).to_dict("records"),
    "causes_annulation": cancel_causes[["cause", "vols", "part_pct"]].round(2).to_dict("records"),
    "h1": {
        "statut": H1_STATUS, "resume": H1_SUMMARY,
        "taux_vacances": round(HOLIDAY_RATE, 2), "taux_hors_vacances": round(BASELINE_RATE, 2),
        "ecart_brut": round(NAIVE_GAP, 2), "cramer_v": H1_TEST["cramer_v"],
        "risque_relatif": H1_TEST["risque_relatif"], "poids_ete": round(SUMMER_WEIGHT, 1),
        "detail": h1_detail.round(2).to_dict("records"),
        "controle": h1_controlled.round(2).to_dict("records"),
        "pires_journees": [{"date": r.DATE.strftime("%d/%m"), "retard": round(float(r.taux_retard), 1),
                            "vols": int(r.vols)} for r in WORST_DELAY_DAYS.itertuples()],
    },
    "h2": {
        "statut": H2_STATUS, "resume": H2_SUMMARY, "composition": H2_COMPO,
        "correlations": H2_CORR,
        "regression": {"pente": round(float(reg.slope), 4), "ordonnee": round(float(reg.intercept), 3)},
        "taux_part_forte": round(HIGH_SHARE_RATE, 2), "taux_part_faible": round(LOW_SHARE_RATE, 2),
        "mediane_part": round(MEDIAN_LONG_SHARE, 2),
        "compagnies": [{"code": r.AIRLINE, "nom": r.compagnie, "vols": int(r.vols),
                        "part_longs": round(float(r.part_vols_longs), 2),
                        "taux_retard": round(float(r.taux_retard), 2),
                        "retard_moyen": round(float(r.retard_moyen), 2),
                        "distance_moyenne": round(float(r.distance_moyenne))}
                       for r in h2_airlines.itertuples()],
        "tranches": [{"tranche": str(r.tranche), "vols": int(r.vols),
                      "taux_retard": round(float(r.taux_retard), 2),
                      "retard_moyen": round(float(r.retard_moyen), 2),
                      "rattrapage": round(float(r.rattrapage_min), 2)}
                     for r in h2_distance.itertuples()],
    },
    "h3": {
        "statut": H3_STATUS, "resume": H3_SUMMARY, "concentration": H3_CONC,
        "part_hiver": round(WINTER_SHARE, 1),
        "lorenz_annulations": downsample(routes_share, cum_cancels),
        "lorenz_vols": downsample(routes_share, cum_traffic),
        "top_routes": [{"rang": int(r.rang), "route": r.ROUTE,
                        "origine": r.origine, "destination": r.destination,
                        "ville_origine": r.ville_origine, "ville_destination": r.ville_destination,
                        "vols": int(r.vols), "annulations": int(r.annulations),
                        "taux": round(float(r.taux_annulation), 2)}
                       for r in TOP_ROUTES.head(10).itertuples()],
        "routes_risquees": [{"route": r.ROUTE, "vols": int(r.vols), "annulations": int(r.annulations),
                             "taux": round(float(r.taux_annulation), 2)}
                            for r in high_rate_routes.head(8).itertuples()],
        "top_aeroports": [{"code": r.aeroport, "vols": int(r.vols), "annulations": int(r.annulations),
                           "taux": round(float(r.taux_annulation), 2)}
                          for r in top_airports.itertuples()],
        # Octobre est hors périmètre (codes aéroport numériques) : on renvoie explicitement
        # une valeur nulle plutôt qu'un mois absent, pour que le dashboard puisse le signaler.
        "saison_prioritaires": [{"mois": m, "taux": (round(float(by_month["taux"].loc[m]), 2)
                                                     if m in by_month.index else None)}
                                for m in range(1, 13)],
    },
    "carte": {"largeur": MAP_W, "hauteur": MAP_H,
              "etats": state_paths, "routes": route_paths, "aeroports": markers},
}

DURATION = round(time.time() - T0, 1)
DASHBOARD_PAYLOAD["meta"]["duree_execution_sec"] = DURATION
SUMMARY["duree_execution_sec"] = DURATION
(OUT / "resume_v2.json").write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8")

target_file = DASH / "dashboard_data.js"
target_file.write_text(
    "// Généré automatiquement par v2/notebook_analyse_v2.ipynb - NE PAS ÉDITER À LA MAIN\n"
    f"// Généré le {SUMMARY['genere_le']}\n"
    "window.DASHBOARD_DATA = "
    + json.dumps(DASHBOARD_PAYLOAD, ensure_ascii=False, separators=(",", ":"))
    + ";\n", encoding="utf-8")

print(f"{target_file.relative_to(PROJECT_ROOT)}  ({target_file.stat().st_size / 1024:.0f} Ko)")
print(f"Durée totale d'exécution : {DURATION} s")

v2/dashboard/dashboard_data.js  (155 Ko)
Durée totale d'exécution : 95.6 s


---
## 11. Synthèse

Le tableau ci-dessous est **généré à partir des variables calculées** : il ne peut pas diverger des résultats.

In [49]:
from IPython.display import Markdown

rows_acc = [
    "| Hypothèse | Verdict | Ce que disent les données |",
    "|---|---|---|",
    f"| **H1** — plus de retards pendant les vacances | **{H1_STATUS}** | {H1_SUMMARY} |",
    f"| **H2** — les longs courriers composent la majorité des retards | **{H2_STATUS}** | {H2_SUMMARY} |",
    f"| **H3** — une minorité de routes concentre les annulations | **{H3_STATUS}** | {H3_SUMMARY} |",
]

conclusion = f"""
### Conclusion générale

Sur {fmt_fr(kpi['vols_programmes'])} vols programmés en 2015, {kpi['taux_retard_pct']} % arrivent avec au moins
{DELAY_THRESHOLD} minutes de retard et {kpi['taux_annulation_pct']} % sont annulés.

Les trois hypothèses ne se valident pas de la même façon, et c'est le principal enseignement de l'analyse :

1. **H1 se confirme, mais pas pour la raison attendue.** Ce ne sont pas « les vacances » qui produisent du
   retard, mais deux pics courts de fin d'année. Thanksgiving, pourtant le plus gros week-end de déplacement
   du pays, n'a aucun effet ({float(h1_controlled.loc[h1_controlled.fenetre.eq('Thanksgiving'), 'effet_net_points'].iloc[0]):+.2f} point).

2. **H2 est réfutée.** Les vols longs ne portent que {H2_COMPO['part_vols_retard_pct']:.1f} % des vols
   en retard — leur part exacte du trafic ({H2_COMPO['part_trafic_pct']:.1f} %). La distance a même un effet
   protecteur (les vols longs rattrapent du retard en vol), et les compagnies spécialisées ne sont pas plus
   en retard (R2 = {H2_CORR['r2']:.3f} sur {H2_CORR['n_compagnies']} compagnies).

3. **H3 se confirme, à condition de la mesurer correctement.** La concentration brute est en grande partie
   mécanique ; le résultat solide est le sur-risque à volume comparable (x{EXCESS_RISK:.2f}), porte par les
   navettes courtes du Nord-Est en hiver.

**Fil conducteur** : retards et annulations répondent à deux logiques différentes. Les retards suivent le
**calendrier** (pics de trafic de fin d'année), les annulations suivent la **météo et la topologie du réseau**
(hiver, Nord-Est, rotations courtes). Un pilotage opérationnel efficace doit donc les traiter séparément —
c'est la lecture que propose le dashboard.
"""

Markdown("\n".join(rows_acc) + "\n" + conclusion)

| Hypothèse | Verdict | Ce que disent les données |
|---|---|---|
| **H1** — plus de retards pendant les vacances | **Confirmée, mais à préciser** | L'écart brut est de +5.03 points (21.91 % contre 16.89 %), mais cet agrégat est dominé à 77 % par l'été et masque des comportements opposés. Une fois chaque fenêtre comparée à une référence comparable, l'effet le plus fort est Nouvel an (+18.25 points vs reste du même mois), l'été ne pèse que +4.12 points, et Thanksgiving est negatif (-0.82 point). Ce ne sont donc pas 'les vacances' qui créent du retard, mais les pics courts de fin d'année. |
| **H2** — les longs courriers composent la majorité des retards | **Réfutée** | Les vols longs (>= 1500 miles) ne composent pas la majorité des retards : ils portent 13.5 % des vols en retard et 13.1 % des minutes de retard, pour 13.6 % des vols effectués. Leur part des retards est donc exactement leur part du trafic : le taux de retard varie à peine avec la distance (18.4 % au-delà de 2000 miles contre 18.2 % en dessous de 500), et les vols les plus longs rattrapent en moyenne 8.9 minutes en vol. La contre-analyse par compagnie écarte aussi la version indirecte de l'hypothèse : sur les 14 compagnies, la corrélation entre part de vols longs et taux de retard est de -0.179 (Pearson, p = 0.54), non significative et de signe contraire à l'hypothèse. |
| **H3** — une minorité de routes concentre les annulations | **Confirmée, avec une réserve importante** | 455 routes sur 4 693 (9.70 %) portent la moitié des 87 430 annulations du périmètre, et le Gini des annulations (0.683) dépasse celui du trafic (0.558). La concentration est donc réelle, mais elle est pour une bonne part mécanique : les 10 % de routes en tête portent 50.9 % des annulations et déjà 38.7 % des vols. Le résultat solide est le sur-risque à volume comparable : 2.92 % d'annulation sur les routes prioritaires contre 1.14 % ailleurs, soit x2.56. Ces routes sont majoritairement des navettes courtes du Nord-Est, et 52 % de leurs annulations tombent au premier trimestre. |

### Conclusion générale

Sur 5 819 079 vols programmés en 2015, 18.61 % arrivent avec au moins
15 minutes de retard et 1.54 % sont annulés.

Les trois hypothèses ne se valident pas de la même façon, et c'est le principal enseignement de l'analyse :

1. **H1 se confirme, mais pas pour la raison attendue.** Ce ne sont pas « les vacances » qui produisent du
   retard, mais deux pics courts de fin d'année. Thanksgiving, pourtant le plus gros week-end de déplacement
   du pays, n'a aucun effet (-0.82 point).

2. **H2 est réfutée.** Les vols longs ne portent que 13.5 % des vols
   en retard — leur part exacte du trafic (13.6 %). La distance a même un effet
   protecteur (les vols longs rattrapent du retard en vol), et les compagnies spécialisées ne sont pas plus
   en retard (R2 = 0.032 sur 14 compagnies).

3. **H3 se confirme, à condition de la mesurer correctement.** La concentration brute est en grande partie
   mécanique ; le résultat solide est le sur-risque à volume comparable (x2.56), porte par les
   navettes courtes du Nord-Est en hiver.

**Fil conducteur** : retards et annulations répondent à deux logiques différentes. Les retards suivent le
**calendrier** (pics de trafic de fin d'année), les annulations suivent la **météo et la topologie du réseau**
(hiver, Nord-Est, rotations courtes). Un pilotage opérationnel efficace doit donc les traiter séparément —
c'est la lecture que propose le dashboard.


---
## 12. Fichiers produits

Tout ce qui suit est régénéré à chaque exécution complète du notebook. Aucun autre script n'est nécessaire.

**`v2/outputs/` — tableaux**

| Fichier | Contenu |
|---|---|
| `kpi_panorama.csv` | indicateurs généraux du dataset |
| `serie_mensuelle.csv`, `serie_journaliere.csv` | taux de retard et d'annulation dans le temps |
| `causes_retard.csv`, `causes_annulation.csv` | décomposition des causes |
| `h1_fenetres_detail.csv` | taux de retard par fenêtre de mobilité |
| `h1_effet_net_controle.csv` | effet net de chaque fenêtre vs sa référence |
| `h2_compagnies.csv` | les 14 compagnies : part de vols longs et ponctualité |
| `h2_tranches_distance.csv` | retard et rattrapage par tranche de distance |
| `h2_composition.csv` | part des vols longs dans le trafic, les vols en retard et les minutes |
| `h3_routes_completes.csv` | les 4 693 routes, avec cumuls de concentration |
| `h3_top_routes.csv`, `h3_routes_plus_risquees.csv`, `h3_top_aeroports.csv` | classements |
| `resume_v2.json` | synthèse chiffrée complète de l'analyse |

**`v2/outputs/figures/` — 6 graphiques PNG** pour le rapport et la presentation.

**`v2/dashboard/`**

| Fichier | Rôle |
|---|---|
| `dashboard_data.js` | généré ici : toutes les données affichées par le dashboard |
| `index.html` | le dashboard, à ouvrir par double-clic (aucun serveur, aucune connexion requise) |

**`v2/assets/states-10m.json`** — fond de carte des États-Unis, téléchargé une seule fois puis mis en cache.